# New Research Questions — RQ1 / RQ2 / RQ5 (self-contained Kaggle driver)

Implements the compute-bearing tasks of `docs/NEW_RQS_TASK_PLAN.md`:

| Phase | Task | What it answers |
|---|---|---|
| **0** | T0 | Checkpoint recovery audit — *are the 120 Step-10 checkpoints recoverable?* |
| **A** | T1 + T2 | RQ1's 2x4 objective x score factorial, and RQ2's VAL-only evidence-affine refit |
| **B** | T5 | RQ5's clean bottleneck-rank sweep (also settles RQ3's causal ambiguity, T3.4/T5.5) |

T3 and T4 are **not** here on purpose: they need no GPU and no checkpoints, and
run locally off `results/mvt_results.json`.

**Kaggle settings:** Accelerator **GPU T4**, Internet **ON**, and attach:
- `beft-thesis-data` (owner `notavailable73`) — CIFAR-100 / SVHN / TinyImageNet / MiniImageNet.
  Everything falls back to a runtime download if it is missing, so this is a speed-up, not a requirement.
- **any Step-10 artifact dataset you saved** — Phase 0 hunts these for the 120 checkpoints.
  See Section 4 for exactly what to attach and how to find it.

**Self-contained:** every piece of *new* logic lives in this notebook (Section 3
writes it out with `%%writefile`, so the source is in the `.ipynb` itself and
travels back in the artifact zip). The repo is still cloned, because the model,
the datasets, the frozen episode seeds, and — critically — `PrototypeHead.to_evidence`
come from it. The evidence map is **never** reimplemented here: that exact
train/eval drift is what caused the Step 4 evidential collapse.

**Validated before shipping.** The factorial evaluator was checked on CPU
against the repo's own `evaluate_episodic` on two real Phase-2 checkpoints:
**13/13 keys exact** on a softmax cell (including `ece_ts`) and **12/12 exact**
on an evidential cell. Section 5 re-proves that in *your* session before any
long run starts.

## 0. Control panel — the only cell you normally edit

Every phase is **resumable**: a cell whose output JSON already exists is
skipped, so re-running this notebook after a session timeout picks up where it
stopped. `MAX_MINUTES_*` finishes the cell in flight and then stops cleanly,
which is what makes a >9 h job survivable in ~9 h sessions.

In [1]:
# ---- What to run this session -------------------------------------------
RUN_PHASE_A = True      # T1 + T2: factorial eval + affine refit over the grid
RUN_PHASE_B = False     # T5: RQ5 bottleneck-rank sweep

# ---- Phase A: which grid cells, and whether to pay for training ----------
# Filter keys: dataset, k_shot, backbone, adapter, head, seed. Empty = all 120.
#   'dataset': 'cifar_fs' | 'mini_imagenet'
#   'backbone': 'resnet18' | 'mobilenetv3_small'
#   'adapter': 'bottleneck_parallel' | 'lora' | 'full_ft' | 'linear_probe'
#
# Empty is the right default given the MEASURED checkpoint coverage (see
# Section 4): 99 of 120 cells were recovered, and the 21 gaps are all in
# cifar_fs/5shot. With ALLOW_RETRAIN=False this evaluates every cell that HAS
# a checkpoint (~5-6 h) and logs the rest as `no_checkpoint` to pick up later.
CELL_FILTER = {}

# False  -> evaluate only cells whose checkpoint was recovered (~3 min each).
# True   -> ALSO train any missing cell first (~18 min each). Run session 1
#           with False; flip to True for a later session to fill the 21 gaps.
ALLOW_RETRAIN = False

# Where Phase 0 hunts for Step 10 artifacts. Add any extra mount point here.
RECOVER_SEARCH_ROOTS = ('/kaggle/input',)

NUM_EPISODES = 600      # the frozen test seeds 0..599; lower ONLY for smoke tests
PERSIST_LOGITS = True   # T1.2 -- ~12 MB/cell, makes all FUTURE re-scoring free
MAX_MINUTES_A = 480.0   # stop cleanly after this long (Kaggle T4 caps ~9 h)

# ---- Phase B: the rank sweep --------------------------------------------
RQ5_HEADS = ('evidential',)         # add 'softmax' to double the runs
RQ5_RANKS = (1, 2, 4, 8, 16, 32, 64)
RQ5_SEEDS = (42, 43, 44)            # 7 x 3 = 21 runs ~ 6 GPU-h
REUSE_GRID_RANK16 = True            # rank 16 IS the Step 10 recipe (n_params 31,746)
MAX_MINUTES_B = 480.0

# ---- Misc ---------------------------------------------------------------
WANDB_MODE = 'disabled'   # 'online' needs a WANDB_API_KEY Kaggle Secret
CACHE_OOD_IMAGES = True   # ~1.2 GB host RAM, saves reloading pools per cell
RUN_SELF_TEST = True      # Section 5 pre-flight; ~3 min, do not skip on run 1

print('phases:', {'A': RUN_PHASE_A, 'B': RUN_PHASE_B},
      '| filter:', CELL_FILTER, '| retrain:', ALLOW_RETRAIN)

phases: {'A': True, 'B': False} | filter: {} | retrain: False


## 1. GPU check + clone repo + install deps

In [2]:
import os, subprocess, sys
import torch

print('python:', sys.version.split()[0], '| torch:', torch.__version__)
print('cuda  :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
if not torch.cuda.is_available():
    print('WARNING: no GPU (Settings > Accelerator > GPU T4). Everything below '
          'still RUNS on CPU, but a 600-episode cell takes hours instead of minutes.')

REPO_URL = 'https://github.com/notAvailable73/thesis.git'
BRANCH   = 'main'
REPO_DIR = '/kaggle/working/thesis' if os.path.isdir('/kaggle/working') else os.path.abspath('./thesis')

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
os.chdir(REPO_DIR)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
sys.path.insert(0, os.getcwd())

from pathlib import Path
REPO = Path(os.getcwd())
print('repo ready at', REPO)
print('HEAD:', subprocess.run(['git', 'log', '-1', '--oneline'],
                              capture_output=True, text=True).stdout.strip())

python: 3.12.13 | torch: 2.10.0+cu128
cuda  : True | Tesla T4


Cloning into '/kaggle/working/thesis'...


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.0/128.0 kB 5.4 MB/s eta 0:00:00
repo ready at /kaggle/working/thesis
HEAD: 5ec6d0a Merge pull request #13 from notAvailable73/Review


## 2. Stage data — symlink the attached Kaggle dataset into `data/`

Nothing here is required: every dataset falls back to a runtime download if its
source is missing (Internet must be ON). This reuses the **same staged-path
finder functions** `scripts/train.py` / `evaluate.py` call at runtime, so it is
guaranteed to point at whatever those modules would discover themselves —
correct regardless of how deep Kaggle happens to mount the dataset.

In [3]:
import shutil

from src.datasets.cifar_fs import _find_staged_cifar100_root
from src.datasets.svhn_ood import _find_staged_svhn_root
from src.datasets.tinyimagenet_ood import _find_extracted_tin_root
from src.datasets.mini_imagenet import _find_zenodo_pkls

LINKS = {}
if (root := _find_staged_cifar100_root('data')):
    LINKS['data/cifar-100-python'] = os.path.join(root, 'cifar-100-python')
if (root := _find_staged_svhn_root('data')):
    LINKS['data/svhn/test_32x32.mat'] = os.path.join(root, 'test_32x32.mat')
if (root := _find_extracted_tin_root('data')):
    LINKS['data/tiny-imagenet-200'] = root
for _split, _pkl in (_find_zenodo_pkls('data') or {}).items():
    LINKS[f'data/{_pkl.name}'] = str(_pkl)

os.makedirs('data/svhn', exist_ok=True)
if not LINKS:
    print('No staged files under /kaggle/input -- did you attach `beft-thesis-data`? '
          'Falling back to runtime downloads for everything.')
for link, target in LINKS.items():
    if os.path.exists(link) or os.path.islink(link):
        print(f'OK   (already present): {link}')
        continue
    if not os.path.exists(target):
        print(f'MISSING source -- {os.path.basename(link)} will download at runtime')
        continue
    try:
        os.symlink(target, link)
        print(f'OK   (symlinked): {link}')
    except OSError as e:
        print(f'symlink failed ({e}); copying instead: {link}')
        (shutil.copytree if os.path.isdir(target) else shutil.copy2)(target, link)

# Frozen splits (idempotent; never regenerates the committed JSON).
subprocess.run([sys.executable, 'scripts/build_cifar_fs_split.py'], check=True)
subprocess.run([sys.executable, 'scripts/build_mini_imagenet_split.py'], check=True)
subprocess.run([sys.executable, 'scripts/build_grid_configs.py'], check=True)
print('\nsplits + 120 grid configs ready')

OK   (symlinked): data/cifar-100-python
OK   (symlinked): data/svhn/test_32x32.mat
OK   (symlinked): data/tiny-imagenet-200
OK   (symlinked): data/mini-imagenet-cache-train.pkl
OK   (symlinked): data/mini-imagenet-cache-validation.pkl
OK   (symlinked): data/mini-imagenet-cache-test.pkl
wrote /kaggle/working/thesis/data/cifar_fs_split.json  (64/16/20, disjoint, union=100, status=canonical_bertinetto_via_torchmeta)
wrote /kaggle/working/thesis/data/mini_imagenet_split.json  (64/16/20, disjoint, union=100, status=canonical_ravi_larochelle)
wrote 120 configs to /kaggle/working/thesis/configs/grid/ (96 PEFT + 24 baseline)
wrote /kaggle/working/thesis/configs/grid/_index.json
  priority 1: 36 cells
  priority 2: 24 cells
  priority 3: 36 cells
  priority 4: 24 cells

splits + 120 grid configs ready


## 3. The new code

Four modules, written out from this notebook so the source lives in the
`.ipynb` and travels back in the artifact zip (Section 10) ready to be merged
into the repo.

- **`rq_core.py`** — the factorised score set, the VAL-only affine refit, the
  T1.6 regression guard, the T2.8 ranking-shift measurement. Its module
  docstring records the T1.0a/b/c design decisions and *why* each was chosen.
- **`rq5_sweep.py`** — RQ5 config generation + the T5.6 "only rank and seed
  differ" guard.
- **`rq_drivers.py`** — Phase 0 recovery/audit, Phase A, Phase B.
- **`rq_aggregate.py`** — the 2x4 tables, the eta-squared variance
  attribution, RQ2's before/after, RQ5's curve.

In [4]:
%%writefile rq_core.py
"""RQ1/RQ2/RQ5 core — factorised objective x score evaluation + VAL-only
evidence-affine refit.

This is the code that is inlined into notebooks/new_rqs.ipynb. It is kept as a
standalone file ONLY so it can be unit-tested on CPU before the Kaggle run; the
notebook carries a verbatim copy.

Design decisions (docs/NEW_RQS_TASK_PLAN.md T1.0a/b/c), all deliberate:

T1.0a  ENERGY on an evidential-trained model is computed on the RAW PROTOTYPE
       LOGITS (pre-`to_evidence`), identically to a softmax-trained model.
       Energy is then the same function of the same quantity in both arms,
       which is exactly what makes the score axis a clean contrast. Computing
       it on alpha would re-confound the two axes.

T1.0b  VACUITY on a softmax-trained model needs an evidence affine, and a
       softmax-trained cell never trained one. We report BOTH:
         vacuity_native  — the checkpoint's own affine. Evidential cells: the
                           TRAINED (scale, bias). Softmax cells: whatever the
                           config left there (base.yaml -> 1.0/0.0). This is
                           the key that must reproduce Step 10 bit-for-bit.
         vacuity_valfit  — (scale, bias) refit on the FROZEN VAL seeds, by the
                           SAME procedure in BOTH arms. This is the headline
                           cross-term, because handing the softmax arm an
                           untuned mapping would stack the comparison exactly
                           the way RQ2 exists to un-stack it.

T1.0c  TS-MSP is well-defined on an evidential-trained model: verified by
       reading scripts/evaluate.py:_fit_val_temperature — it consumes
       `model.forward_proto_from_features` logits and calls `fit_temperature`,
       and never branches on `interpretation`. evaluate.py merely declined to
       CALL it for evidential cells; the function itself is logit-level and
       interpretation-agnostic.

T2.2   The evidence map is NEVER reimplemented here. Every softplus goes
       through `PrototypeHead.to_evidence`, and every evidence->prob/vacuity
       step through `src.evaluators.ood.evidence_to_probs_and_vacuity`. The
       Step 4 collapse happened because train and eval evidence maps drifted.
"""
from __future__ import annotations

import json
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import yaml

from src.datasets import (
    EpisodicIterableDataset, get_id_split, get_svhn_ood, get_heldout_near_ood,
    get_tinyimagenet_ood, get_gaussian_ood,
)
from src.datasets.mini_imagenet import MINI_IMAGENET_ALL_WNIDS
from src.evaluators import (
    accuracy, f1_macro, expected_calibration_error, brier_score,
    ood_auroc, fpr_at_95_tpr, energy_score, fit_temperature,
    evidence_to_probs_and_vacuity,
)
from src.evaluators.temperature import apply_temperature
from src.heads.prototype_head import PrototypeHead
from src.models import build_model

# Reused verbatim from the repo so the OOD-feature path cannot drift.
from scripts.evaluate import _extract_features

#: The four scores of RQ1's score axis, plus the native-affine vacuity kept
#: alongside it as the Step-10 regression anchor / sensitivity row.
FACTORIAL_SCORES = ("msp", "energy", "ts_msp", "vacuity_valfit", "vacuity_native")

# ---------------------------------------------------------------------------
# Process-lifetime caches. A grid sweep evaluates dozens of cells that share
# the same in-distribution splits and the same OOD image pools; only the
# BACKBONE differs, so the images can be loaded once and re-featurised per
# cell. The ID splits are lazy torchvision Datasets (cheap). The OOD pools are
# materialised (N, 3, 224, 224) float tensors -- ~300 MB each at n=500 -- so
# caching all four costs ~1.2 GB of host RAM; set CACHE_OOD_IMAGES=False if a
# session is memory-constrained.
# ---------------------------------------------------------------------------
_SPLIT_CACHE: dict = {}
_OOD_IMG_CACHE: dict = {}
CACHE_OOD_IMAGES = True


def cached_id_split(dataset_cfg, split: str):
    key = (str(dataset_cfg.get("name", "cifar_fs")), split,
           int(dataset_cfg.get("image_size", 224)),
           str(dataset_cfg.get("data_root", "data")),
           json.dumps(list(dataset_cfg.get("class_ids") or []), sort_keys=True))
    if key not in _SPLIT_CACHE:
        _SPLIT_CACHE[key] = get_id_split(dataset_cfg, split=split)
    return _SPLIT_CACHE[key]


def _cached_ood_images(key, builder):
    if not CACHE_OOD_IMAGES:
        return builder()
    if key not in _OOD_IMG_CACHE:
        _OOD_IMG_CACHE[key] = builder()
    return _OOD_IMG_CACHE[key]


def clear_caches():
    _SPLIT_CACHE.clear()
    _OOD_IMG_CACHE.clear()


# =====================================================================
# Evidence affine: read, override, refit
# =====================================================================
def read_evidence_affine(head: PrototypeHead) -> tuple[float, float]:
    """The (scale, bias) the checkpoint actually carries.

    `scale` is read through the head's own parameterisation (softplus of the
    raw parameter when learnable) rather than recomputed, so it matches what
    `to_evidence` will use.
    """
    with torch.no_grad():
        if head.evidence_affine:
            scale = float(F.softplus(head._evidence_raw_scale).item())
            bias = float(head._evidence_bias.item())
        else:
            scale = float(head._evidence_scale_const.item())
            bias = float(head._evidence_bias_const.item())
    return scale, bias


@contextmanager
def evidence_affine_override(head: PrototypeHead, scale: float, bias: float):
    """Temporarily install (scale, bias) on the head itself.

    Deliberately mutates the head's own parameters/buffers instead of adding a
    second evidence path, so `head.to_evidence` remains the single source of
    truth (T2.2). Restores the originals on exit, including on exception.
    """
    with torch.no_grad():
        if head.evidence_affine:
            old = (head._evidence_raw_scale.detach().clone(),
                   head._evidence_bias.detach().clone())
            raw = torch.log(torch.expm1(torch.tensor(
                max(float(scale), 1e-4), dtype=head._evidence_raw_scale.dtype)))
            head._evidence_raw_scale.copy_(raw.to(head._evidence_raw_scale.device))
            head._evidence_bias.copy_(torch.tensor(
                float(bias), dtype=head._evidence_bias.dtype,
                device=head._evidence_bias.device))
        else:
            old = (head._evidence_scale_const.detach().clone(),
                   head._evidence_bias_const.detach().clone())
            head._evidence_scale_const.copy_(torch.tensor(
                float(scale), dtype=head._evidence_scale_const.dtype,
                device=head._evidence_scale_const.device))
            head._evidence_bias_const.copy_(torch.tensor(
                float(bias), dtype=head._evidence_bias_const.dtype,
                device=head._evidence_bias_const.device))
    try:
        yield
    finally:
        with torch.no_grad():
            if head.evidence_affine:
                head._evidence_raw_scale.copy_(old[0])
                head._evidence_bias.copy_(old[1])
            else:
                head._evidence_scale_const.copy_(old[0])
                head._evidence_bias_const.copy_(old[1])


def fit_evidence_affine(val_logits: torch.Tensor, val_targets: torch.Tensor,
                        *, num_classes: int, prior_per_class: float,
                        scale_init: float, bias_init: float,
                        max_iter: int = 500, lr: float = 0.05) -> tuple[float, float]:
    """RQ2 / T2.1 — refit the two evidence-affine scalars on VAL logits only.

    Mirrors scripts/evaluate.py:_fit_val_temperature's protocol exactly: one
    global fit, on the frozen VAL episodes, minimising NLL (Guo et al. 2017),
    fixed iteration count, no randomness -> deterministic.

    The optimisation runs on a THROWAWAY PrototypeHead whose only role is to
    own two learnable scalars, so `to_evidence` is still the one softplus in
    the codebase. The caller never has to touch the real head's gradients.
    """
    logits = val_logits.detach().float().cpu()
    targets = val_targets.detach().long().cpu()

    fit_head = PrototypeHead(
        metric="l2",  # unused: only to_evidence() is called on this head
        evidence_affine=True,
        evidence_scale_init=max(float(scale_init), 1e-4),
        evidence_bias_init=float(bias_init),
    )
    params = [fit_head._evidence_raw_scale, fit_head._evidence_bias]
    opt = torch.optim.Adam(params, lr=lr)
    for _ in range(max_iter):
        opt.zero_grad()
        evidence = fit_head.to_evidence(logits)
        probs, _vac = evidence_to_probs_and_vacuity(
            evidence, num_classes, prior_per_class)
        loss = F.nll_loss(torch.log(probs.clamp_min(1e-12)), targets)
        loss.backward()
        opt.step()
    return read_evidence_affine(fit_head)


# =====================================================================
# The factorised score set (T1.1)
# =====================================================================
def all_id_scores(logits: torch.Tensor, head: PrototypeHead, *,
                  num_classes: int, temperature: float | None,
                  prior_per_class: float,
                  affine_valfit: tuple[float, float]) -> dict[str, torch.Tensor]:
    """Every score for BOTH objectives, driven by an explicit list rather than
    an `if interpretation ==` branch (T1.1). Higher => more in-distribution.

    Replaces src/evaluators/episodic.py:_id_score_set's diagonal-only behaviour
    WITHOUT changing it: on a softmax cell msp/energy/ts_msp are computed from
    the identical expressions, and on an evidential cell vacuity_native goes
    through the identical `head.to_evidence` call, so Step 10's numbers are
    reproduced exactly (verified by the T1.6 regression guard).
    """
    scores: dict[str, torch.Tensor] = {}

    probs = torch.softmax(logits, dim=-1)
    scores["msp"] = probs.max(dim=-1).values
    # T1.0a: raw prototype logits in BOTH arms.
    scores["energy"] = energy_score(logits)
    if temperature is not None:
        scores["ts_msp"] = apply_temperature(logits, temperature).max(dim=-1).values

    # Native affine: whatever the checkpoint carries.
    evidence = head.to_evidence(logits)
    _p, vac = evidence_to_probs_and_vacuity(evidence, num_classes, prior_per_class)
    scores["vacuity_native"] = 1.0 - vac

    # VAL-refit affine, same procedure in both arms (T1.0b option ii).
    with evidence_affine_override(head, *affine_valfit):
        evidence_v = head.to_evidence(logits)
    _pv, vac_v = evidence_to_probs_and_vacuity(evidence_v, num_classes, prior_per_class)
    scores["vacuity_valfit"] = 1.0 - vac_v

    # The head's evidence affine is a learnable Parameter on evidential cells,
    # so these carry grad when called outside torch.no_grad(). Scoring is a
    # read-only operation; detaching makes the function safe to call anywhere
    # (e.g. post-hoc re-scoring of a persisted logit dump).
    return {k: v.detach() for k, v in scores.items()}


def all_prob_sets(logits: torch.Tensor, head: PrototypeHead, *,
                  num_classes: int, temperature: float | None,
                  prior_per_class: float,
                  affine_valfit: tuple[float, float]) -> dict[str, torch.Tensor]:
    """Probability vectors under every objective, for ECE / Brier / F1.

    `evidential_native` on an evidential cell is Step 10's `ece_pooled`;
    `evidential_valfit` is RQ2's "after". `softmax` / `ts` are the softmax arm.
    """
    out = {"softmax": torch.softmax(logits, dim=-1)}
    if temperature is not None:
        out["ts"] = apply_temperature(logits, temperature)

    evidence = head.to_evidence(logits)
    out["evidential_native"], _ = evidence_to_probs_and_vacuity(
        evidence, num_classes, prior_per_class)

    with evidence_affine_override(head, *affine_valfit):
        evidence_v = head.to_evidence(logits)
    out["evidential_valfit"], _ = evidence_to_probs_and_vacuity(
        evidence_v, num_classes, prior_per_class)
    return {k: v.detach() for k, v in out.items()}


# =====================================================================
# Data paths (replicated from scripts/evaluate.py, not re-invented)
# =====================================================================
def load_val_logits(model, cfg, device, repo_root: Path):
    """Pooled VAL query logits + targets.

    Byte-for-byte the same data path as scripts/evaluate.py:_fit_val_temperature
    (same file, same seeds, same iterable construction), so a temperature fit
    on top of this reproduces the one Step 10 recorded.

    T2.5 guard: returns the val seed list it actually used so the caller can
    assert it is [10000..10099] and disjoint from the 600 test seeds. The test
    split is never constructed in this function.
    """
    with open(repo_root / "configs" / "val_episodes.yaml") as f:
        val_spec = yaml.safe_load(f)
    val_seeds = list(val_spec["seeds"])
    val_split = cached_id_split(cfg.dataset, "val")
    val_iter = EpisodicIterableDataset(
        val_split, n_way=int(cfg.dataset.n_way), k_shot=int(cfg.dataset.k_shot),
        q_query=int(cfg.dataset.q_query), num_episodes=len(val_seeds),
        seed_offset=int(val_seeds[0]),
    )
    backbone = model.backbone
    logits_all, targets_all = [], []
    model.eval()
    with torch.no_grad():
        for sx, sy, qx, qy in val_iter:
            sf = backbone(sx.to(device))
            qf = backbone(qx.to(device))
            ql = model.forward_proto_from_features(sf, sy.to(device), qf)
            logits_all.append(ql.cpu())
            targets_all.append(qy.cpu())
    return torch.cat(logits_all), torch.cat(targets_all), val_seeds


def build_ood_pools(model, cfg, device, *, use_tinyimagenet=True, use_gaussian=True):
    """The same four pools, in the same insertion order, as the Step 10 grid
    invocation (`--use-tinyimagenet --use-gaussian`). Order matters: the first
    pool is the legacy `primary_ood_pool` (svhn_far)."""
    img_size = int(cfg.dataset.image_size)
    n_ood = int(cfg.ood.num_samples)
    ood_seed = int(cfg.ood.seed)
    pools = {}

    ds_name = str(cfg.dataset.get("name", "cifar_fs"))
    base_key = (ds_name, img_size, n_ood, ood_seed)

    svhn_x = _cached_ood_images(
        ("svhn", img_size, n_ood, ood_seed),
        lambda: get_svhn_ood(data_root=cfg.ood.data_root, image_size=img_size,
                             num_samples=n_ood, seed=ood_seed))
    pools["svhn_far"] = _extract_features(model.backbone, svhn_x, device)

    near_name, heldout_x = _cached_ood_images(
        ("heldout",) + base_key,
        lambda: get_heldout_near_ood(cfg.dataset, num_samples=n_ood,
                                     seed=ood_seed, heldout_split="val"))
    pools[near_name] = _extract_features(model.backbone, heldout_x, device)

    if use_tinyimagenet:
        # Step 9: TinyImageNet-200 shares 25 wnids with MiniImageNet's 100
        # classes, so an "OOD" pool could otherwise contain literal ID images.
        exclude = (MINI_IMAGENET_ALL_WNIDS if ds_name == "mini_imagenet" else None)
        tin_x = _cached_ood_images(
            ("tin",) + base_key,
            lambda: get_tinyimagenet_ood(data_root=cfg.dataset.data_root,
                                         image_size=img_size, num_samples=n_ood,
                                         seed=ood_seed, exclude_wnids=exclude))
        pools["tin_near"] = _extract_features(model.backbone, tin_x, device)

    if use_gaussian:
        gauss_x = _cached_ood_images(
            ("gauss", img_size, n_ood, ood_seed),
            lambda: get_gaussian_ood(image_size=img_size, num_samples=n_ood,
                                     seed=ood_seed))
        pools["gaussian_far"] = _extract_features(model.backbone, gauss_x, device)

    return pools


# =====================================================================
# The factorial evaluation
# =====================================================================
def factorial_evaluate(model, cfg, *, test_seeds, ood_pools, device,
                       temperature, affine_valfit, prior_per_class,
                       ece_bins=15, logits_out: Path | None = None,
                       log_every=100, logger_print=print) -> dict:
    """Run the 600 test episodes once and score them under EVERY
    (objective, score) combination.

    Returns a summary dict whose native-interpretation keys are numerically
    identical to Step 10's, plus the cross-terms. Optionally persists the raw
    per-episode logits (T1.2) so every FUTURE post-hoc scoring question is a
    re-analysis rather than a retrain.
    """
    K = int(cfg.dataset.n_way)
    head = model.head
    n_eval = len(test_seeds)
    seed_offset = int(test_seeds[0])
    if test_seeds != list(range(seed_offset, seed_offset + n_eval)):
        raise ValueError("test seeds must be a contiguous range (see "
                         "scripts/evaluate.py:_evaluate_episodic)")

    test_split = cached_id_split(cfg.dataset, "test")
    test_iter = EpisodicIterableDataset(
        test_split, n_way=K, k_shot=int(cfg.dataset.k_shot),
        q_query=int(cfg.dataset.q_query), num_episodes=n_eval,
        seed_offset=seed_offset,
    )

    pool_names = list(ood_pools.keys())
    ood_pools = {k: v.to(device) for k, v in ood_pools.items()}
    prob_sets = ("softmax", "ts", "evidential_native", "evidential_valfit")

    per_ep = {f"acc__{p}": [] for p in prob_sets}
    per_ep.update({f"f1__{p}": [] for p in prob_sets})
    per_ep.update({f"ece__{p}": [] for p in prob_sets})
    per_ep.update({f"brier__{p}": [] for p in prob_sets})
    auroc_acc = {p: {s: [] for s in FACTORIAL_SCORES} for p in pool_names}
    fpr_acc = {p: {s: [] for s in FACTORIAL_SCORES} for p in pool_names}
    pooled = {p: [] for p in prob_sets}
    pooled_targets = []
    pooled_logits = []   # needed to reproduce ece_ts EXACTLY -- see below

    dump_id, dump_ood, dump_tgt = [], {p: [] for p in pool_names}, []
    # T2.8: keep both vacuity variants so "does the OOD ranking survive the
    # refit?" is answered with a measured number per pool, not an assertion.
    track_id = {"vacuity_native": [], "vacuity_valfit": []}
    track_ood = {p: {"vacuity_native": [], "vacuity_valfit": []} for p in pool_names}

    model.eval()
    backbone = model.backbone
    with torch.no_grad():
        for i, (sx, sy, qx, qy) in enumerate(test_iter):
            sx, sy = sx.to(device), sy.to(device)
            qx, qy = qx.to(device), qy.to(device)
            sf = backbone(sx)
            qf = backbone(qx)
            q_logits = model.forward_proto_from_features(sf, sy, qf)

            probs = all_prob_sets(q_logits, head, num_classes=K,
                                  temperature=temperature,
                                  prior_per_class=prior_per_class,
                                  affine_valfit=affine_valfit)
            for name, p in probs.items():
                per_ep[f"acc__{name}"].append(accuracy(p, qy))
                per_ep[f"f1__{name}"].append(f1_macro(p, qy, num_classes=K))
                per_ep[f"ece__{name}"].append(
                    expected_calibration_error(p, qy, num_bins=ece_bins))
                per_ep[f"brier__{name}"].append(brier_score(p, qy, K))
                pooled[name].append(p.cpu())
            pooled_targets.append(qy.cpu())
            pooled_logits.append(q_logits.cpu())

            id_scores = all_id_scores(q_logits, head, num_classes=K,
                                      temperature=temperature,
                                      prior_per_class=prior_per_class,
                                      affine_valfit=affine_valfit)
            for k in track_id:
                track_id[k].append(id_scores[k].cpu().numpy())
            if logits_out is not None:
                dump_id.append(q_logits.cpu().to(torch.float16).numpy())
                dump_tgt.append(qy.cpu().to(torch.int8).numpy())

            for pname in pool_names:
                ood_logits = model.forward_proto_from_features(
                    sf, sy, ood_pools[pname])
                ood_scores = all_id_scores(ood_logits, head, num_classes=K,
                                           temperature=temperature,
                                           prior_per_class=prior_per_class,
                                           affine_valfit=affine_valfit)
                for k in track_ood[pname]:
                    track_ood[pname][k].append(ood_scores[k].cpu().numpy())
                if logits_out is not None:
                    dump_ood[pname].append(
                        ood_logits.cpu().to(torch.float16).numpy())
                for sname in FACTORIAL_SCORES:
                    if sname not in id_scores:
                        continue
                    id_np = id_scores[sname].cpu().numpy()
                    ood_np = ood_scores[sname].cpu().numpy()
                    auroc_acc[pname][sname].append(ood_auroc(id_np, ood_np))
                    fpr_acc[pname][sname].append(fpr_at_95_tpr(id_np, ood_np))

            if log_every and (i + 1) % log_every == 0:
                logger_print(f"    ep {i + 1}/{n_eval}  "
                             f"acc={np.mean(per_ep['acc__softmax']):.4f}")

    summary: dict = {"num_episodes": n_eval, "n_way": K}
    for name in prob_sets:
        if not per_ep[f"acc__{name}"]:
            continue
        pooled_p = torch.cat(pooled[name], dim=0)
        tgt = torch.cat(pooled_targets, dim=0)
        summary[f"accuracy_mean__{name}"] = float(np.mean(per_ep[f"acc__{name}"]))
        summary[f"accuracy_std__{name}"] = float(np.std(per_ep[f"acc__{name}"]))
        summary[f"f1_macro_mean__{name}"] = float(np.mean(per_ep[f"f1__{name}"]))
        summary[f"ece_per_episode_mean__{name}"] = float(np.mean(per_ep[f"ece__{name}"]))
        summary[f"ece_pooled__{name}"] = float(
            expected_calibration_error(pooled_p, tgt, num_bins=ece_bins))
        summary[f"brier_mean__{name}"] = float(np.mean(per_ep[f"brier__{name}"]))

    # `ece_ts` is the ONE key src/evaluators/episodic.py computes from the
    # CONCATENATED logits (`apply_temperature(pooled_logits_t, T)`) instead of
    # per-episode-then-concatenate like every other pooled metric there.
    # Softmax is row-wise, so the two are mathematically identical -- but
    # torch.softmax picks a different CUDA kernel for (75, K) than for
    # (E*75, K), and the differing reduction order surfaces at ~1e-8. Invisible
    # on CPU, real on a T4. Reproduce the repo's exact expression so the T1.6
    # guard can still report `exact`, rather than widening a tolerance until it
    # would also hide a genuine regression.
    if temperature is not None and pooled_logits:
        pooled_logits_t = torch.cat(pooled_logits, dim=0)
        tgt_all = torch.cat(pooled_targets, dim=0)
        ts_pooled = apply_temperature(pooled_logits_t, temperature)
        summary["ece_pooled__ts"] = float(expected_calibration_error(
            ts_pooled, tgt_all, num_bins=ece_bins))
        # The repo's `brier_ts` is likewise a single POOLED Brier, not the mean
        # of per-episode Briers `brier_mean__ts` reports. Keep both: they are
        # different quantities and only this one is comparable to Step 10.
        summary["brier_pooled__ts"] = float(brier_score(ts_pooled, tgt_all, K))

    for pname in pool_names:
        for sname in FACTORIAL_SCORES:
            vals = auroc_acc[pname][sname]
            if not vals:
                continue
            summary[f"ood_auroc__{pname}__{sname}"] = float(np.mean(vals))
            summary[f"ood_auroc_std__{pname}__{sname}"] = float(np.std(vals))
            summary[f"fpr_at_95_tpr__{pname}__{sname}"] = float(
                np.mean(fpr_acc[pname][sname]))

    # T2.8 — measured, per pool, over the exact ID+OOD vector AUROC ranks.
    id_nat = np.concatenate(track_id["vacuity_native"])
    id_fit = np.concatenate(track_id["vacuity_valfit"])
    for pname in pool_names:
        a = np.concatenate([id_nat, np.concatenate(track_ood[pname]["vacuity_native"])])
        b = np.concatenate([id_fit, np.concatenate(track_ood[pname]["vacuity_valfit"])])
        summary[f"ranking_shift__{pname}"] = ranking_shift(a, b)

    if logits_out is not None:
        logits_out.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(
            logits_out,
            id_logits=np.stack(dump_id),          # (E, Q, K) float16
            id_targets=np.stack(dump_tgt),        # (E, Q)    int8
            pool_names=np.array(pool_names),
            **{f"ood_logits__{p}": np.stack(dump_ood[p]) for p in pool_names},
        )
        summary["logits_dump"] = str(logits_out)
        summary["logits_dump_bytes"] = int(logits_out.stat().st_size)

    return summary


# =====================================================================
# T1.6 regression guard
# =====================================================================
def native_score_name(interpretation: str) -> str:
    return "vacuity_native" if interpretation == "evidential" else "msp"


def regression_guard(summary: dict, committed_path: Path, interpretation: str,
                     *, tol: float = 1e-6) -> dict:
    """T1.6 — prove the refactor added cross-terms without perturbing the
    diagonal, by diffing against the COMMITTED Step 10 metrics JSON.

    Checks every key Step 10 wrote for this cell's native interpretation, and
    grades the result in three tiers rather than pass/fail:

      exact       every key bit-identical. Expected when re-running on the SAME
                  hardware the committed numbers came from. Measured: 12/12 and
                  13/13 exact when the old and new evaluators run in one process.
      within_tol  max |diff| <= tol. This is a PASS. Re-running the committed
                  cifar_1shot/mbnet/lora cell on CPU against numbers produced on
                  a Kaggle T4 gives max |diff| = 2.2e-7 -- float32 accumulation
                  differing across devices, which flips a handful of near-tied
                  ID/OOD pairs and moves a 600-episode mean AUROC in the 7th
                  decimal. Nothing about the logic changed.
      MISMATCH    max |diff| > tol: a real difference, investigate before
                  quoting anything.

    Why 1e-6 and not something tighter: a genuine logic error here -- wrong
    score, wrong affine, wrong pool -- moves AUROC by 1e-2 to 1e-1, four to five
    orders of magnitude above the float32 noise floor. A threshold in between
    separates them cleanly without ever calling hardware noise a defect.
    """
    if not committed_path.exists():
        return {"status": "no_committed_file", "path": str(committed_path)}
    old = json.load(open(committed_path))
    native = native_score_name(interpretation)
    new_prob_set = "evidential_native" if interpretation == "evidential" else "softmax"

    checks: dict[str, tuple[float, float]] = {}
    for key, val in old.items():
        if key.startswith("ood_auroc__") or key.startswith("fpr_at_95_tpr__"):
            head, pool, score = key.split("__")
            if score != ("vacuity" if interpretation == "evidential" else "msp"):
                continue
            checks[key] = (float(val), summary.get(f"{head}__{pool}__{native}", float("nan")))
    for old_key, new_key in (
        ("accuracy_mean", f"accuracy_mean__{new_prob_set}"),
        ("ece_pooled", f"ece_pooled__{new_prob_set}"),
        ("brier_mean", f"brier_mean__{new_prob_set}"),
        ("f1_macro_mean", f"f1_macro_mean__{new_prob_set}"),
        ("ece_ts", "ece_pooled__ts"),
    ):
        if old_key in old and new_key in summary:
            checks[old_key] = (float(old[old_key]), float(summary[new_key]))

    diffs = {k: abs(a - b) for k, (a, b) in checks.items()}
    if not diffs:
        return {"status": "no_comparable_keys", "path": str(committed_path)}
    max_key = max(diffs, key=diffs.get)
    n_exact = sum(1 for d in diffs.values() if d == 0.0)
    return {
        "status": ("exact" if n_exact == len(diffs)
                   else "within_tol" if diffs[max_key] <= tol else "MISMATCH"),
        "tol": tol,
        "n_keys": len(diffs),
        "n_exact": n_exact,
        "max_abs_diff": float(diffs[max_key]),
        "max_abs_diff_key": max_key,
        "path": str(committed_path),
    }


# =====================================================================
# T2.8 — is vacuity reordering even possible under an affine change?
# =====================================================================
def ranking_shift(score_a: np.ndarray, score_b: np.ndarray,
                  *, max_pairs: int = 2_000_000, seed: int = 0) -> dict:
    """Turn RQ2's central question into a measurement.

    `softplus(a*x + b)` is monotone in each logit separately, but vacuity is
    K / sum_k(alpha_k) — a function of the SUM. A monotone per-logit map is not
    monotone in the sum, so reordering is possible in principle. This reports
    whether it happens in practice: Spearman rho and the exact fraction of
    sample pairs whose relative order flips.
    """
    a = np.asarray(score_a, dtype=np.float64).ravel()
    b = np.asarray(score_b, dtype=np.float64).ravel()
    n = len(a)
    ra = np.argsort(np.argsort(a))
    rb = np.argsort(np.argsort(b))
    rho = float(np.corrcoef(ra, rb)[0, 1]) if n > 1 else 1.0

    total_pairs = n * (n - 1) // 2
    if total_pairs <= max_pairs:
        da = a[:, None] - a[None, :]
        db = b[:, None] - b[None, :]
        iu = np.triu_indices(n, k=1)
        disc = int(np.sum(np.sign(da[iu]) * np.sign(db[iu]) < 0))
        sampled = total_pairs
    else:
        rng = np.random.default_rng(seed)
        i = rng.integers(0, n, size=max_pairs)
        j = rng.integers(0, n, size=max_pairs)
        keep = i != j
        i, j = i[keep], j[keep]
        disc = int(np.sum(np.sign(a[i] - a[j]) * np.sign(b[i] - b[j]) < 0))
        sampled = len(i)
    return {
        "spearman_rho": rho,
        "discordant_pairs": disc,
        "pairs_compared": int(sampled),
        "discordant_fraction": float(disc / sampled) if sampled else 0.0,
        "exhaustive": total_pairs <= max_pairs,
        "reordering_observed": disc > 0,
    }

Writing rq_core.py


In [5]:
%%writefile rq5_sweep.py
"""RQ5 / T5.3-T5.6 — the clean bottleneck-rank sweep.

Removes the confound in the existing four budget points (linear_probe / LoRA /
bottleneck-parallel / full_ft), which vary budget TOGETHER WITH adapter type
and with which weights train. Here dataset (CIFAR-FS), backbone (ResNet-18),
shot count and adapter family (parallel 1x1 bottleneck) are all held fixed and
ONLY `adapter.rank` moves.

This is also the experiment that settles RQ3's causal ambiguity (T3.4): it
varies budget with backbone AND architecture fixed, which the Step 10 grid
cannot do.
"""
from __future__ import annotations

import json
from pathlib import Path

import yaml

from src.utils import load_config
from scripts.train import _head_descriptor, _checkpoint_tag

#: ~7 budget points spanning 2.9k -> 124k trainable params on ResNet-18.
#: 16 is the grid's frozen value, so the sweep passes exactly through the
#: existing Step 10 point rather than running parallel to it.
RANKS = (1, 2, 4, 8, 16, 32, 64)
SWEEP_SEEDS = (42, 43, 44)

PARENTS = {
    "evidential": "exp_phase3_placement_parallel_evidential.yaml",
    "softmax": "exp_phase3_placement_parallel_softmax.yaml",
}

#: Keys this generator is ALLOWED to differ on across the sweep. Everything
#: else must be byte-identical (T5.6) or the sweep silently reintroduces the
#: very confound it exists to remove.
ALLOWED_DIFF_PATHS = {
    "seed",
    "adapter.rank",
    "output.run_tag",
    "output.results_dir",
    "wandb.group",
    "wandb.tags",
    "wandb.mode",
    "wandb.disabled",
}


def sweep_cell_id(head: str, rank: int, seed: int) -> str:
    return f"rq5_r18_parallel_{head}_rank{rank}_seed{seed}"


def build_sweep_configs(repo_root: Path, heads=("evidential",),
                        ranks=RANKS, seeds=SWEEP_SEEDS,
                        out_dir_name="rq5", results_dir="results/rq5") -> list[dict]:
    """Write one YAML per (head, rank, seed) and return the index.

    Each file is a 5-key override on the SAME Step 6 parent config the grid
    used, so the recipe (LR, KL schedule, evidence affine, block_ids, epochs,
    patience) is inherited, never retyped.
    """
    cfg_dir = repo_root / "configs" / out_dir_name
    cfg_dir.mkdir(parents=True, exist_ok=True)
    cells = []
    for head in heads:
        parent = PARENTS[head]
        for rank in ranks:
            for seed in seeds:
                cell = sweep_cell_id(head, rank, seed)
                run_tag = f"rq5_r18_parallel_rank{rank}"
                doc = {
                    "extends": f"../{parent}",
                    "seed": int(seed),
                    "adapter": {"rank": int(rank)},
                    "output": {"run_tag": run_tag,
                               "results_dir": results_dir},
                    "wandb": {"group": f"rq5-rank-sweep-{head}",
                              "tags": ["rq5", "rank_sweep", head]},
                }
                path = cfg_dir / f"{cell}.yaml"
                with open(path, "w") as f:
                    f.write(
                        "# RQ5 rank sweep (T5.3) — generated by the new_rqs "
                        "notebook, do not hand-edit.\n"
                        f"# Only adapter.rank and seed vary across this sweep; "
                        f"see the T5.6 guard.\n")
                    yaml.safe_dump(doc, f, sort_keys=False)

                merged = load_config(path)
                results_suffix = f"{cell}"
                cells.append({
                    "cell": cell, "head": head, "rank": int(rank), "seed": int(seed),
                    "config": str(path.relative_to(repo_root)),
                    "run_tag": run_tag,
                    "results_suffix": results_suffix,
                    "checkpoint": (f"checkpoints/model_phase2_"
                                   f"{_checkpoint_tag(merged)}.pt"),
                    "results_json": (f"{results_dir}/{results_suffix}_"
                                     f"{merged.adapter.type}_"
                                     f"{_head_descriptor(merged)}_metrics.json"),
                })
    index_path = cfg_dir / "_index.json"
    with open(index_path, "w") as f:
        json.dump({"cells": cells}, f, indent=2, sort_keys=True)
    return cells


# =====================================================================
# T5.6 — the guard that keeps the sweep a controlled experiment
# =====================================================================
def _flatten(d, prefix=""):
    out = {}
    for k, v in d.items():
        path = f"{prefix}.{k}" if prefix else str(k)
        if isinstance(v, dict):
            out.update(_flatten(v, path))
        else:
            out[path] = v
    return out


def assert_only_rank_and_seed_differ(repo_root: Path, cells: list[dict]) -> dict:
    """T5.6 — assert the MERGED configs differ only in rank, seed and the
    derived output/wandb bookkeeping.

    Checks the merged result, not the YAML source: a stray difference inherited
    from a parent would be invisible in the 5-key override files.
    """
    merged = {}
    for c in cells:
        merged[c["cell"]] = _flatten(dict(load_config(repo_root / c["config"])))

    keys = set()
    for m in merged.values():
        keys |= set(m.keys())

    offenders = {}
    for key in sorted(keys):
        if key in ALLOWED_DIFF_PATHS:
            continue
        vals = {json.dumps(m.get(key, "<MISSING>"), sort_keys=True, default=str)
                for m in merged.values()}
        if len(vals) > 1:
            offenders[key] = sorted(vals)[:4]

    return {
        "n_configs": len(cells),
        "n_keys_checked": len(keys - ALLOWED_DIFF_PATHS),
        "ok": not offenders,
        "offending_keys": offenders,
    }


def expected_trainable_params(rank: int,
                              stage_channels=(64, 128, 256, 512),
                              n_affine: int = 2) -> int:
    """Closed-form check on the parallel 1x1 bottleneck budget.

    Per stage: down 1x1 conv (C->r, +r bias) and up 1x1 conv (r->C, +C bias).
    Cross-checked against the committed grid value n_params=31746 at rank 16
    (ResNet-18, evidential: 2 extra evidence-affine scalars).
    """
    total = sum(2 * c * rank + rank + c for c in stage_channels)
    return total + n_affine

Writing rq5_sweep.py


In [6]:
%%writefile rq_drivers.py
"""Phase 0 / A / B drivers — checkpoint recovery, factorial evaluation, rank sweep.

Every driver is resumable (skips a cell whose output JSON already exists) and
budget-aware (`max_minutes` finishes the current cell then stops cleanly), for
the same reason scripts/run_mvt_grid.py is: a hosted-notebook session can be
killed at any moment and nothing already paid for should be lost.
"""
from __future__ import annotations

import json
import os
import shutil
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import torch

import rq_core as R
from src.utils import load_config, set_seed, count_trainable_params
from src.models import build_model
from src.evaluators import fit_temperature
from scripts.evaluate import _load_test_seeds

#: The frozen VAL seeds. Hard-coded here ONLY as an assertion target: the
#: fitting code reads configs/val_episodes.yaml, and this is the tripwire that
#: proves it never silently drifted onto the 600 test seeds (T2.5).
EXPECTED_VAL_SEED_RANGE = (10000, 10099)


# =====================================================================
# Phase 0 — T0 checkpoint recovery audit
# =====================================================================
def recover_checkpoints(repo_root: Path, search_roots=("/kaggle/input",),
                        log=print) -> dict:
    """Find Step 10 checkpoints in whatever the session has attached.

    Handles both shapes the Step 10 notebooks could have produced:
      - the artifact ZIPs those notebooks pushed (step10a/b/c packed
        `checkpoints/model_phase2_*.pt` alongside `results/grid/*`), and
      - loose `.pt` files, if a dataset was built by copying rather than zipping.

    ZIPs are extracted to a staging directory and only `.pt` files are copied
    into `checkpoints/`. Committed `results/` files are deliberately NOT
    overwritten: they are the T1.6 regression baseline, and clobbering them
    with a copy of themselves would destroy the only independent check we have.
    """
    ckpt_dir = repo_root / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    staging = repo_root / "_recovered"
    staging.mkdir(parents=True, exist_ok=True)

    found_zips, loose_pt, copied, skipped = [], [], [], []
    for root in search_roots:
        rp = Path(root)
        if not rp.exists():
            log(f"  (no such path: {root})")
            continue
        for p in sorted(rp.rglob("*")):
            if not p.is_file():
                continue
            if p.suffix == ".zip":
                found_zips.append(p)
            elif p.suffix == ".pt" and p.name.startswith("model_phase2_"):
                loose_pt.append(p)

    log(f"  scanned {search_roots}: {len(found_zips)} zip(s), "
        f"{len(loose_pt)} loose checkpoint(s)")

    for z in found_zips:
        try:
            with zipfile.ZipFile(z) as zf:
                members = [m for m in zf.namelist()
                           if m.startswith("checkpoints/") and m.endswith(".pt")]
                if not members:
                    log(f"  {z.name}: no checkpoints/ members, skipped")
                    continue
                # Extract only the members we want, and never outside staging.
                for m in members:
                    dest = (staging / m).resolve()
                    if not str(dest).startswith(str(staging.resolve())):
                        log(f"  !! refusing unsafe zip member {m!r} in {z.name}")
                        continue
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    with zf.open(m) as src, open(dest, "wb") as out:
                        shutil.copyfileobj(src, out)
                log(f"  {z.name}: extracted {len(members)} checkpoint(s)")
        except zipfile.BadZipFile:
            log(f"  {z.name}: not a readable zip, skipped")

    candidates = list(staging.rglob("model_phase2_*.pt")) + loose_pt
    for src in candidates:
        dest = ckpt_dir / src.name
        if dest.exists():
            skipped.append(dest.name)
            continue
        shutil.copy2(src, dest)
        copied.append(dest.name)

    return {"zips_seen": [str(p) for p in found_zips],
            "loose_pt_seen": len(loose_pt),
            "copied": copied, "already_present": skipped}


def audit_checkpoints(repo_root: Path, log=print) -> dict:
    """T0.3 — the written verdict, computed rather than assumed."""
    index = json.load(open(repo_root / "configs" / "grid" / "_index.json"))["cells"]
    present, missing = [], []
    for c in index:
        (present if (repo_root / c["checkpoint"]).exists() else missing).append(c)

    by_seed: dict[int, int] = {}
    by_slice: dict[str, list[int]] = {}
    for c in present:
        by_seed[c["seed"]] = by_seed.get(c["seed"], 0) + 1
    for c in index:
        key = f"{c['dataset']}/{c['k_shot']}shot"
        got = by_slice.setdefault(key, [0, 0])
        got[1] += 1
        if (repo_root / c["checkpoint"]).exists():
            got[0] += 1

    n = len(present)
    seeds_present = sorted(by_seed)
    if n == len(index):
        verdict, note = "a", "All 120 recovered — RQ1/RQ2 are evaluation-only."
    elif n == 0:
        verdict, note = "c", ("None recovered — RQ1 needs the grid retrain; RQ2 "
                              "piggybacks on it rather than costing extra.")
    elif seeds_present == [42] and n == 40:
        verdict, note = "b", ("Seed-42 subset only — RQ1/RQ2 run at n=1 seed. "
                              "Report WITHOUT seed error bars and say so.")
    else:
        verdict, note = "b-partial", (f"Partial recovery ({n}/120). Usable, but "
                                      f"state the coverage explicitly.")

    log(f"\n  checkpoints present: {n}/{len(index)}")
    log(f"  by seed: {dict(sorted(by_seed.items())) or '(none)'}")
    for k in sorted(by_slice):
        got, tot = by_slice[k]
        log(f"    {k:<26} {got}/{tot}")
    log(f"\n  VERDICT ({verdict}): {note}")

    return {"n_present": n, "n_total": len(index), "verdict": verdict,
            "note": note, "by_seed": by_seed,
            "by_slice": {k: v for k, v in sorted(by_slice.items())},
            "missing_cells": [c["cell" if "cell" in c else "config"] for c in missing]}


# =====================================================================
# One factorial evaluation
# =====================================================================
def _rel(path, root: Path) -> str:
    """Repo-relative path, falling back to absolute.

    `Path.relative_to` RAISES when the target is outside `root`, and this is
    called while assembling the result record -- i.e. AFTER a 600-episode
    evaluation has already been paid for. Step 11's post-mortem
    (step_writeups/step11.txt Section 8) is exactly this: a crash in
    bookkeeping threw away a correct measurement before it could be saved.
    Bookkeeping must never be able to destroy a result.
    """
    try:
        return str(Path(path).relative_to(root))
    except ValueError:
        return str(Path(path).resolve())


def factorial_run_one(repo_root: Path, config_path: Path, checkpoint_path: Path,
                      out_json: Path, *, device, num_episodes: int,
                      logits_out: Path | None, committed_metrics: Path | None,
                      meta: dict | None = None, log=print) -> dict:
    """Evaluate one trained cell under every (objective, score) combination."""
    t0 = time.monotonic()
    cfg = load_config(config_path)
    interp = cfg.head.get("interpretation", "evidential")
    K = int(cfg.dataset.n_way)
    prior_pc = float(cfg.loss.get("prior_per_class", 1.0))

    set_seed(int(cfg.seed))
    model = build_model(cfg).to(device)
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["state_dict"])
    n_params = int(count_trainable_params(model))
    affine_native = R.read_evidence_affine(model.head)

    # --- VAL-only fitting (T2.1 / T2.5) -------------------------------
    val_logits, val_targets, val_seeds = R.load_val_logits(
        model, cfg, device, repo_root)
    lo, hi = EXPECTED_VAL_SEED_RANGE
    if not (val_seeds[0] == lo and val_seeds[-1] == hi):
        raise RuntimeError(
            f"VAL seed guard failed: fitting would use seeds "
            f"{val_seeds[0]}..{val_seeds[-1]}, expected {lo}..{hi}. "
            f"Refusing to continue — this is the repo's hard convention.")
    test_seeds_all = _load_test_seeds(repo_root, cfg)
    if set(val_seeds) & set(test_seeds_all):
        raise RuntimeError("VAL and TEST seed sets overlap — refusing to fit.")

    T = fit_temperature(val_logits, val_targets)
    affine_valfit = R.fit_evidence_affine(
        val_logits, val_targets, num_classes=K, prior_per_class=prior_pc,
        scale_init=affine_native[0], bias_init=affine_native[1])
    log(f"    T={T:.4f}  affine native=({affine_native[0]:.4f}, "
        f"{affine_native[1]:.4f})  refit=({affine_valfit[0]:.4f}, "
        f"{affine_valfit[1]:.4f})")

    pools = R.build_ood_pools(model, cfg, device,
                              use_tinyimagenet=True, use_gaussian=True)
    test_seeds = test_seeds_all[:num_episodes]
    summary = R.factorial_evaluate(
        model, cfg, test_seeds=test_seeds, ood_pools=pools, device=device,
        temperature=T, affine_valfit=affine_valfit, prior_per_class=prior_pc,
        ece_bins=int(cfg.eval.ece_bins), logits_out=logits_out,
        log_every=200, logger_print=log)

    guard = (R.regression_guard(summary, committed_metrics, interp)
             if committed_metrics else {"status": "not_requested"})

    record = {
        "meta": meta or {},
        "config": _rel(config_path, repo_root),
        "checkpoint": _rel(checkpoint_path, repo_root),
        "interpretation": interp,
        "adapter_type": cfg.adapter.type,
        "adapter_rank": int(cfg.adapter.get("rank", -1)),
        "adapter_placement": str(cfg.adapter.get("placement", "post_pool")),
        "backbone": str(cfg.backbone.name),
        "dataset": str(cfg.dataset.get("name", "cifar_fs")),
        "k_shot": int(cfg.dataset.k_shot),
        "seed": int(cfg.seed),
        "n_params": n_params,
        "best_val_epoch": int(ckpt.get("best_val_epoch", -1)),
        "temperature": float(T),
        "affine_native": [float(affine_native[0]), float(affine_native[1])],
        "affine_valfit": [float(affine_valfit[0]), float(affine_valfit[1])],
        "affine_config_init": [float(cfg.head.get("evidence_scale_init", 1.0)),
                               float(cfg.head.get("evidence_bias_init", 0.0))],
        "val_seeds": {"first": int(val_seeds[0]), "last": int(val_seeds[-1]),
                      "n": len(val_seeds)},
        "summary": summary,
        "regression_guard": guard,
        "wall_seconds": round(time.monotonic() - t0, 1),
    }
    out_json.parent.mkdir(parents=True, exist_ok=True)
    with open(out_json, "w") as f:
        json.dump(record, f, indent=2, sort_keys=True)
    return record


def train_cell(config_path: Path, wandb_mode="disabled", log=print) -> None:
    """Train one cell by calling the REAL scripts/train.py main() in-process.

    In-process, not subprocess: Step 9 found subprocess output can silently
    vanish on hosted notebooks (scripts/run_mvt_grid.py's docstring), and this
    keeps exactly one source of truth for what "a run" means.
    """
    import scripts.train as train_mod
    old = sys.argv
    sys.argv = ["train.py", "--config", str(config_path), "--wandb-mode", wandb_mode]
    try:
        train_mod.main()
    finally:
        sys.argv = old


# =====================================================================
# Phase A — RQ1 + RQ2 over the grid
# =====================================================================
def run_phase_a(repo_root: Path, cells: list[dict], *, device, out_dir: Path,
                logits_dir: Path | None, num_episodes: int, allow_retrain: bool,
                wandb_mode: str, max_minutes: float | None, log=print) -> dict:
    out_dir.mkdir(parents=True, exist_ok=True)
    run_log = out_dir / "_run_log.jsonl"
    started = time.monotonic()
    counts = {"ok": 0, "skipped_done": 0, "no_checkpoint": 0, "error": 0,
              "trained": 0}

    for i, c in enumerate(cells):
        if max_minutes is not None and (time.monotonic() - started) / 60 > max_minutes:
            log(f"[A] budget {max_minutes} min reached; "
                f"{len(cells) - i} cell(s) left for the next session.")
            break

        cell_id = (f"{c['dataset']}_{c['k_shot']}shot_{c['backbone']}_"
                   f"{c['adapter']}_{c['head']}_seed{c['seed']}")
        out_json = out_dir / f"{cell_id}.json"
        if out_json.exists():
            counts["skipped_done"] += 1
            continue

        log(f"[A] ({i + 1}/{len(cells)}) {cell_id}")
        ckpt = repo_root / c["checkpoint"]
        try:
            if not ckpt.exists():
                if not allow_retrain:
                    log("    no checkpoint and ALLOW_RETRAIN=False -> skipped")
                    counts["no_checkpoint"] += 1
                    _append(run_log, {"cell": cell_id, "status": "no_checkpoint"})
                    continue
                log("    no checkpoint -> training this cell first")
                train_cell(repo_root / c["config"], wandb_mode=wandb_mode, log=log)
                counts["trained"] += 1

            rec = factorial_run_one(
                repo_root, repo_root / c["config"], ckpt, out_json,
                device=device, num_episodes=num_episodes,
                logits_out=(logits_dir / f"{cell_id}.npz") if logits_dir else None,
                committed_metrics=repo_root / c["results_json"],
                meta=c, log=log)
            g = rec["regression_guard"]
            log(f"    guard: {g.get('status')} "
                f"({g.get('n_exact', 0)}/{g.get('n_keys', 0)} exact, "
                f"max|diff|={g.get('max_abs_diff', float('nan')):.2e})"
                f"  [{rec['wall_seconds']}s]")
            counts["ok"] += 1
            _append(run_log, {"cell": cell_id, "status": "ok",
                              "guard": g.get("status"),
                              "wall_seconds": rec["wall_seconds"]})
        except Exception as e:  # noqa: BLE001 — one bad cell must not kill the phase
            log(f"    ERROR: {e!r}")
            counts["error"] += 1
            _append(run_log, {"cell": cell_id, "status": "error", "error": repr(e)})

    log(f"[A] done: {counts}")
    return counts


# =====================================================================
# Phase B — RQ5 rank sweep
# =====================================================================
def run_phase_b(repo_root: Path, cells: list[dict], *, device, out_dir: Path,
                logits_dir: Path | None, num_episodes: int, wandb_mode: str,
                max_minutes: float | None, reuse_grid_rank16: bool = True,
                log=print) -> dict:
    out_dir.mkdir(parents=True, exist_ok=True)
    run_log = out_dir / "_run_log.jsonl"
    started = time.monotonic()
    counts = {"ok": 0, "skipped_done": 0, "trained": 0, "reused_grid": 0, "error": 0}

    grid_index = json.load(open(repo_root / "configs/grid/_index.json"))["cells"]

    for i, c in enumerate(cells):
        if max_minutes is not None and (time.monotonic() - started) / 60 > max_minutes:
            log(f"[B] budget {max_minutes} min reached; "
                f"{len(cells) - i} cell(s) left for the next session.")
            break

        out_json = out_dir / f"{c['cell']}.json"
        if out_json.exists():
            counts["skipped_done"] += 1
            continue

        log(f"[B] ({i + 1}/{len(cells)}) {c['cell']}  (rank={c['rank']}, seed={c['seed']})")
        ckpt = repo_root / c["checkpoint"]
        try:
            if not ckpt.exists() and reuse_grid_rank16 and c["rank"] == 16:
                # rank 16 + parallel + r18 + CIFAR-FS 5-shot IS the Step 10 grid
                # recipe (verified: n_params 31,746 both ways). If that cell's
                # checkpoint was recovered, retraining it would burn ~18 min to
                # reproduce a model we already have.
                twin = _find_grid_twin(grid_index, c)
                if twin and (repo_root / twin["checkpoint"]).exists():
                    shutil.copy2(repo_root / twin["checkpoint"], ckpt)
                    log(f"    reused Step 10 checkpoint: {twin['checkpoint']}")
                    counts["reused_grid"] += 1

            if not ckpt.exists():
                log("    training…")
                train_cell(repo_root / c["config"], wandb_mode=wandb_mode, log=log)
                counts["trained"] += 1

            rec = factorial_run_one(
                repo_root, repo_root / c["config"], ckpt, out_json,
                device=device, num_episodes=num_episodes,
                logits_out=(logits_dir / f"{c['cell']}.npz") if logits_dir else None,
                committed_metrics=None, meta=c, log=log)
            log(f"    acc={rec['summary'].get('accuracy_mean__softmax', float('nan')):.4f} "
                f"ECE(evid)={rec['summary'].get('ece_pooled__evidential_native', float('nan')):.4f} "
                f"params={rec['n_params']}  [{rec['wall_seconds']}s]")
            counts["ok"] += 1
            _append(run_log, {"cell": c["cell"], "status": "ok",
                              "wall_seconds": rec["wall_seconds"]})
        except Exception as e:  # noqa: BLE001
            log(f"    ERROR: {e!r}")
            counts["error"] += 1
            _append(run_log, {"cell": c["cell"], "status": "error", "error": repr(e)})

    log(f"[B] done: {counts}")
    return counts


def _find_grid_twin(grid_index: list[dict], sweep_cell: dict) -> dict | None:
    for g in grid_index:
        if (g["dataset"] == "cifar_fs" and g["k_shot"] == 5
                and g["backbone"] == "resnet18"
                and g["adapter"] == "bottleneck_parallel"
                and g["head"] == sweep_cell["head"]
                and g["seed"] == sweep_cell["seed"]):
            return g
    return None


def _append(path: Path, entry: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    entry = {**entry, "ts": time.strftime("%Y-%m-%dT%H:%M:%S")}
    with open(path, "a") as f:
        f.write(json.dumps(entry, sort_keys=True) + "\n")

Writing rq_drivers.py


In [7]:
%%writefile rq_aggregate.py
"""Aggregation — RQ1's 2x4 factorial + variance attribution, RQ2's
before/after, RQ5's rank curves.

Reads only the per-cell JSONs the drivers wrote, so it can be re-run locally
after the session without a GPU.
"""
from __future__ import annotations

import json
from itertools import combinations
from pathlib import Path

import numpy as np

#: RQ1's score axis. `vacuity_valfit` is the headline vacuity (T1.0b option ii:
#: the affine is refit on VAL by the SAME procedure in both arms).
#: `vacuity_native` is carried as a sensitivity row, not a factorial column.
RQ1_SCORES = ("msp", "energy", "ts_msp", "vacuity_valfit")
FAR_POOLS = ("svhn_far", "gaussian_far")
NEAR_POOLS = ("cifar100_near", "mini_near", "tin_near")


def load_records(out_dir: Path) -> list[dict]:
    recs = []
    for p in sorted(Path(out_dir).glob("*.json")):
        if p.name.startswith("_"):
            continue
        recs.append(json.load(open(p)))
    return recs


def design_key(rec: dict) -> str:
    return (f"{rec['dataset']}/{rec['k_shot']}shot/{rec['backbone']}/"
            f"{rec['meta'].get('adapter', rec['adapter_type'])}")


# =====================================================================
# RQ1
# =====================================================================
def factorial_observations(recs: list[dict]) -> list[dict]:
    """One row per (design, objective, score, pool, seed) -> AUROC."""
    rows = []
    for r in recs:
        s = r["summary"]
        for key, val in s.items():
            if not key.startswith("ood_auroc__"):
                continue
            _, pool, score = key.split("__")
            if score not in RQ1_SCORES:
                continue
            rows.append({
                "design": design_key(r),
                "objective": r["interpretation"],
                "score": score,
                "pool": pool,
                "pool_group": ("far" if pool in FAR_POOLS else
                               "near" if pool in NEAR_POOLS else "other"),
                "seed": r["seed"],
                "auroc": float(val),
            })
    return rows


def table_2x4(rows: list[dict], pool_group: str | None = None) -> dict:
    """The 2 (objective) x 4 (score) matrix RQ1 exists to fill in.

    Only the diagonal of this table existed before: evidential cells were
    scored with vacuity only, softmax cells with msp/energy/ts_msp only.
    """
    sel = [r for r in rows if pool_group is None or r["pool_group"] == pool_group]
    out = {}
    for obj in ("evidential", "softmax"):
        out[obj] = {}
        for sc in RQ1_SCORES:
            vals = [r["auroc"] for r in sel
                    if r["objective"] == obj and r["score"] == sc]
            out[obj][sc] = {
                "mean": float(np.mean(vals)) if vals else float("nan"),
                "std": float(np.std(vals)) if vals else float("nan"),
                "n": len(vals),
            }
    return out


def eta_squared(rows: list[dict], factors: list[str], value: str = "auroc",
                interactions: bool = True) -> dict:
    """Classical SS decomposition -> eta^2 per factor.

    Exact for a balanced design; the returned `balanced` flag says whether the
    design actually is balanced, because the formula quietly stops being exact
    when it is not, and a silently-wrong variance attribution is precisely the
    failure mode T4.1 flagged in the existing decomposition.
    """
    y = np.array([r[value] for r in rows], dtype=float)
    if len(y) == 0:
        return {"error": "no observations"}
    grand = y.mean()
    ss_total = float(((y - grand) ** 2).sum())
    levels = {f: np.array([r[f] for r in rows], dtype=object) for f in factors}

    cell_counts: dict[tuple, int] = {}
    for i in range(len(y)):
        key = tuple(levels[f][i] for f in factors)
        cell_counts[key] = cell_counts.get(key, 0) + 1
    balanced = len(set(cell_counts.values())) == 1

    def group_means(keys):
        idx: dict[tuple, list[int]] = {}
        for i in range(len(y)):
            k = tuple(levels[f][i] for f in keys)
            idx.setdefault(k, []).append(i)
        return {k: (y[v].mean(), len(v)) for k, v in idx.items()}

    ss: dict[str, float] = {}
    main = {f: group_means([f]) for f in factors}
    for f in factors:
        ss[f] = float(sum(n * (m - grand) ** 2 for m, n in main[f].values()))

    if interactions:
        for a, b in combinations(factors, 2):
            gm = group_means([a, b])
            tot = 0.0
            for (la, lb), (m, n) in gm.items():
                ma = main[a][(la,)][0]
                mb = main[b][(lb,)][0]
                tot += n * (m - ma - mb + grand) ** 2
            ss[f"{a}:{b}"] = float(tot)

    ss_resid = ss_total - sum(ss.values())
    out = {k: (v / ss_total if ss_total else float("nan")) for k, v in ss.items()}
    out["residual"] = ss_resid / ss_total if ss_total else float("nan")
    return {"eta_squared": out, "ss_total": ss_total, "n": int(len(y)),
            "balanced": balanced,
            "cell_counts": sorted(set(cell_counts.values()))}


def rq1_verdict(rows: list[dict]) -> dict:
    """Does the OBJECTIVE or the SCORE explain more of the AUROC variance?"""
    out = {}
    for group in ("far", "near"):
        sel = [r for r in rows if r["pool_group"] == group]
        if not sel:
            continue
        eta = eta_squared(sel, ["design", "objective", "score"])
        e = eta["eta_squared"]
        out[group] = {
            "eta_squared": e,
            "dominant": ("score" if e.get("score", 0) > e.get("objective", 0)
                         else "objective"),
            "ratio_score_over_objective": (
                e.get("score", 0) / e["objective"]
                if e.get("objective", 0) > 0 else float("inf")),
            "balanced": eta["balanced"],
            "n": eta["n"],
        }
    return out


# =====================================================================
# RQ2
# =====================================================================
def rq2_table(recs: list[dict]) -> list[dict]:
    """Per evidential cell: ECE and OOD-AUROC before/after the VAL refit.

    The headline is the JOINT outcome — an ECE improvement only matters if the
    OOD ranking survives it, which is why every row carries both.
    """
    rows = []
    for r in recs:
        if r["interpretation"] != "evidential":
            continue
        s = r["summary"]
        row = {
            "cell": r["meta"].get("cell") or design_key(r) + f"/seed{r['seed']}",
            "design": design_key(r), "seed": r["seed"],
            "ece_before": s.get("ece_pooled__evidential_native"),
            "ece_after": s.get("ece_pooled__evidential_valfit"),
            "ece_softmax_ref": s.get("ece_pooled__softmax"),
            "ece_ts_ref": s.get("ece_pooled__ts"),
            "brier_before": s.get("brier_mean__evidential_native"),
            "brier_after": s.get("brier_mean__evidential_valfit"),
            "affine_trained": r["affine_native"],
            "affine_refit": r["affine_valfit"],
            "affine_config_init": r["affine_config_init"],
        }
        row["ece_delta"] = ((row["ece_after"] - row["ece_before"])
                            if None not in (row["ece_after"], row["ece_before"])
                            else None)
        for key in s:
            if key.startswith("ood_auroc__") and key.endswith("__vacuity_native"):
                pool = key.split("__")[1]
                before = s[key]
                after = s.get(f"ood_auroc__{pool}__vacuity_valfit")
                row[f"auroc_before__{pool}"] = before
                row[f"auroc_after__{pool}"] = after
                if after is not None:
                    row[f"auroc_delta__{pool}"] = after - before
            if key.startswith("ranking_shift__"):
                row[f"rank_rho__{key.split('__')[1]}"] = s[key]["spearman_rho"]
                row[f"rank_discordant__{key.split('__')[1]}"] = \
                    s[key]["discordant_fraction"]
        rows.append(row)
    return rows


def rq2_verdict(rows: list[dict]) -> dict:
    if not rows:
        return {"error": "no evidential cells"}
    ece_d = [r["ece_delta"] for r in rows if r["ece_delta"] is not None]
    auroc_d = [v for r in rows for k, v in r.items()
               if k.startswith("auroc_delta__") and v is not None]
    rho = [v for r in rows for k, v in r.items() if k.startswith("rank_rho__")]
    return {
        "n_cells": len(rows),
        "ece_improved_in": int(sum(1 for d in ece_d if d < 0)),
        "ece_mean_delta": float(np.mean(ece_d)) if ece_d else None,
        "auroc_preserved_in": int(sum(1 for d in auroc_d if d >= -0.005)),
        "auroc_comparisons": len(auroc_d),
        "auroc_mean_delta": float(np.mean(auroc_d)) if auroc_d else None,
        "min_spearman_rho": float(np.min(rho)) if rho else None,
        "reordering_ever_observed": bool(any(
            r.get(k, 0) > 0 for r in rows for k in r
            if k.startswith("rank_discordant__"))),
    }


# =====================================================================
# RQ5
# =====================================================================
def rq5_curve(recs: list[dict]) -> dict:
    """Accuracy / ECE / AUROC vs trainable-parameter budget at FIXED
    architecture — the curve that tells RQ5's U-shape from an artefact of
    varying the adapter type at the same time."""
    by_rank: dict[int, dict] = {}
    for r in recs:
        rank = r["adapter_rank"]
        s = r["summary"]
        e = by_rank.setdefault(rank, {"rank": rank, "n_params": r["n_params"],
                                      "seeds": [], "acc": [], "ece_evid": [],
                                      "ece_softmax": [], "ece_ts": [],
                                      "auroc_far": [], "auroc_near": []})
        e["seeds"].append(r["seed"])
        e["acc"].append(s.get("accuracy_mean__softmax"))
        e["ece_evid"].append(s.get("ece_pooled__evidential_native"))
        e["ece_softmax"].append(s.get("ece_pooled__softmax"))
        e["ece_ts"].append(s.get("ece_pooled__ts"))
        far = [s[f"ood_auroc__{p}__vacuity_native"] for p in FAR_POOLS
               if f"ood_auroc__{p}__vacuity_native" in s]
        near = [s[f"ood_auroc__{p}__vacuity_native"] for p in NEAR_POOLS
                if f"ood_auroc__{p}__vacuity_native" in s]
        e["auroc_far"].append(float(np.mean(far)) if far else None)
        e["auroc_near"].append(float(np.mean(near)) if near else None)

    curve = []
    for rank in sorted(by_rank):
        e = by_rank[rank]
        row = {"rank": rank, "n_params": e["n_params"], "n_seeds": len(e["seeds"])}
        for k in ("acc", "ece_evid", "ece_softmax", "ece_ts",
                  "auroc_far", "auroc_near"):
            vals = [v for v in e[k] if v is not None]
            row[f"{k}_mean"] = float(np.mean(vals)) if vals else None
            row[f"{k}_std"] = float(np.std(vals)) if vals else None
        curve.append(row)

    def _argopt(key, mode):
        pts = [(r["rank"], r[key]) for r in curve if r.get(key) is not None]
        if not pts:
            return None
        return (min if mode == "min" else max)(pts, key=lambda t: t[1])[0]

    best_acc_rank = _argopt("acc_mean", "max")
    best_ece_rank = _argopt("ece_evid_mean", "min")
    ranks = [r["rank"] for r in curve]
    return {
        "curve": curve,
        "best_accuracy_rank": best_acc_rank,
        "best_ece_rank": best_ece_rank,
        "budgets_differ": (best_acc_rank != best_ece_rank),
        "ece_optimum_is_interior": (
            best_ece_rank not in (min(ranks), max(ranks)) if ranks else None),
    }

Writing rq_aggregate.py


In [8]:
import importlib, json, time
import rq_core, rq5_sweep, rq_drivers, rq_aggregate
for _m in (rq_core, rq5_sweep, rq_drivers, rq_aggregate):
    importlib.reload(_m)
import rq_core as R
import rq5_sweep as S
import rq_drivers as D
import rq_aggregate as A

R.CACHE_OOD_IMAGES = CACHE_OOD_IMAGES
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Tee everything to disk. The Step 11 post-mortem (step_writeups/step11.txt
# Section 8) is explicit: a crash in a hosted notebook left NO trace in the
# saved .ipynb and was only diagnosable because a side-channel log happened to
# survive. Make that luck into a guarantee.
SESSION_LOG = REPO / 'results' / 'new_rqs_session.log'
SESSION_LOG.parent.mkdir(parents=True, exist_ok=True)

def log(*args):
    line = ' '.join(str(a) for a in args)
    print(line, flush=True)
    with open(SESSION_LOG, 'a') as f:
        f.write(line + '\n')

log(f'=== session start {time.strftime("%Y-%m-%d %H:%M:%S")} device={DEVICE} ===')
print('modules loaded; scores =', R.FACTORIAL_SCORES)

=== session start 2026-08-25 09:25:18 device=cuda ===
modules loaded; scores = ('msp', 'energy', 'ts_msp', 'vacuity_valfit', 'vacuity_native')


## 4. Phase 0 — checkpoint recovery audit (T0)

**This is the cell that decides the schedule.** RQ1 and RQ2 both need trained
adapters. The difference between "recovered" and "gone" is ~2 days of
evaluation versus ~36 GPU-hours of retraining, so it gets settled with evidence
before anything long starts.

It scans everything under `RECOVER_SEARCH_ROOTS`, extracts `checkpoints/*.pt`
out of any Step-10 artifact zip it finds (and also picks up loose `.pt` files,
since Kaggle sometimes auto-unzips an uploaded dataset), then enumerates all
120 cells in `configs/grid/_index.json` to report exactly which are present.

**Committed `results/` files are never overwritten** — they are the T1.6
regression baseline, and a copy of themselves landing on top would destroy the
only independent check we have that the refactor didn't move Step 10's numbers.

### Measured coverage (checked against the three artifact zips, 2026-08-25)

| slice | recovered |
|---|---|
| `cifar_fs/1shot` | **36/36** |
| `cifar_fs/5shot` | **15/36** |
| `mini_imagenet/1shot` | **24/24** |
| `mini_imagenet/5shot` | **24/24** |
| **total** | **99/120**, all three seeds |

The 21 gaps are all `cifar_fs/5shot` PEFT cells (parallel + LoRA on both
backbones) — consistent with that session running under disk pressure. All six
adapter types were confirmed to `load_state_dict(strict=True)` and to match
their committed `n_params`, so the recovered checkpoints are sound.

**Verdict (b-partial): RQ1 and RQ2 are evaluation-only for 99/120 cells.** No
retrain is needed to answer either question — only to close the 21-cell gap.

In [9]:
audit_before = D.audit_checkpoints(REPO, log=log)

log('\n--- scanning attached datasets for Step 10 artifacts ---')
recovery = D.recover_checkpoints(REPO, search_roots=RECOVER_SEARCH_ROOTS, log=log)
log(f"  copied {len(recovery['copied'])} checkpoint(s) into checkpoints/")

log('\n--- audit after recovery ---')
audit = D.audit_checkpoints(REPO, log=log)
audit['recovery'] = recovery
with open(REPO / 'results' / 'rq_checkpoint_audit.json', 'w') as f:
    json.dump(audit, f, indent=2, sort_keys=True)

if audit['n_present'] == 0:
    log("""
  ================= NOTHING RECOVERED — WHAT TO DO =================
  The Step 10a/b/c notebooks packed `checkpoints/model_phase2_grid_*.pt` into
  the artifact zip they pushed. 99 of 120 are known to exist. Two ways in:

    A. ZERO UPLOAD -- attach the original notebook outputs directly:
       + Add Input  ->  Notebooks tab  ->  find step10a / step10b / step10c
       ->  Add.  Their output mounts under /kaggle/input/<notebook-slug>/.
       Add all three, re-run this cell. Nothing else changes.

    B. Upload the three zips as one private Kaggle dataset (~2.65 GB):
       kaggle.com/datasets  ->  New Dataset  ->  drag
       step10a_cifar_5shot_artifacts.zip, step10b_cifar_1shot_artifacts.zip,
       step10c_mini_artifacts.zip  ->  Create.  Then + Add Input it here.
       (Auto-unzip on or off both work -- this cell handles either.)

  Only if BOTH fail is this verdict (c): set ALLOW_RETRAIN = True and let
  Phase A train from scratch at ~18 min/cell.
  ==================================================================""")
elif audit['verdict'] in ('b', 'b-partial'):
    log('\n  NOTE: partial recovery. Whatever you report from this must state '
        'the seed coverage explicitly -- with one seed there are no seed error '
        'bars, and saying so is the difference between a limitation and a flaw.')


  checkpoints present: 0/120
  by seed: (none)
    cifar_fs/1shot             0/36
    cifar_fs/5shot             0/36
    mini_imagenet/1shot        0/24
    mini_imagenet/5shot        0/24

  VERDICT (c): None recovered — RQ1 needs the grid retrain; RQ2 piggybacks on it rather than costing extra.

--- scanning attached datasets for Step 10 artifacts ---
  scanned ('/kaggle/input',): 0 zip(s), 99 loose checkpoint(s)
  copied 99 checkpoint(s) into checkpoints/

--- audit after recovery ---

  checkpoints present: 99/120
  by seed: {42: 33, 43: 33, 44: 33}
    cifar_fs/1shot             36/36
    cifar_fs/5shot             15/36
    mini_imagenet/1shot        24/24
    mini_imagenet/5shot        24/24

  VERDICT (b-partial): Partial recovery (99/120). Usable, but state the coverage explicitly.

  NOTE: partial recovery. Whatever you report from this must state the seed coverage explicitly -- with one seed there are no seed error bars, and saying so is the difference between a limitatio

## 5. Pre-flight self-test — fail fast, before burning hours

Five checks, ~3 minutes, **no checkpoint required** (it builds an untrained
model, which is enough to prove the scoring paths agree):

1. **T1.6 — the diagonal is untouched.** Runs the repo's own
   `evaluate_episodic` and the new `factorial_evaluate` over the same 3
   episodes with the same pools, and requires the native-interpretation keys to
   agree **exactly**. This is what proves RQ1 cannot silently invalidate Step 10.
2. **T1.5** — every score is produced for *both* interpretations.
3. **T1.7** — energy on an evidential model equals `logsumexp` of the same raw
   logits a softmax model would use (i.e. T1.0a is really implemented).
4. **T1.8 / T2.6** — two identical runs produce byte-identical JSON.
5. **T5.6 + the parameter formula** — the 21 sweep configs differ only in rank
   and seed, and the built parameter counts match the closed form (rank 16 must
   land on **31,746**, the value `results/mvt_results.json` recorded).

If any of these fails, **stop and read the output** — do not start Phase A.

In [10]:
if not RUN_SELF_TEST:
    print('RUN_SELF_TEST = False -- skipped.')
else:
    import numpy as np
    from src.utils import load_config, set_seed
    from src.models import build_model
    from src.evaluators import evaluate_episodic, fit_temperature
    from src.datasets import EpisodicIterableDataset

    failures = []
    cfg_e = load_config(REPO / 'configs/grid/cifar_5shot_r18_parallel_evidential_seed42.yaml')
    cfg_s = load_config(REPO / 'configs/grid/cifar_5shot_r18_parallel_softmax_seed42.yaml')
    for c in (cfg_e, cfg_s):
        c.ood['num_samples'] = 40          # smoke-test size; both paths see the same pools
    N_SMOKE = 3

    def _smoke(cfg, tag):
        interp = cfg.head.get('interpretation', 'evidential')
        K = int(cfg.dataset.n_way)
        prior = float(cfg.loss.get('prior_per_class', 1.0))
        set_seed(int(cfg.seed))
        model = build_model(cfg).to(DEVICE)     # UNTRAINED: enough to compare paths
        pools = R.build_ood_pools(model, cfg, DEVICE)

        test_split = R.cached_id_split(cfg.dataset, 'test')
        it = EpisodicIterableDataset(test_split, n_way=K, k_shot=int(cfg.dataset.k_shot),
                                     q_query=int(cfg.dataset.q_query),
                                     num_episodes=N_SMOKE, seed_offset=0)
        vl, vt, vseeds = R.load_val_logits(model, cfg, DEVICE, REPO)
        assert vseeds[0] == 10000 and vseeds[-1] == 10099, 'VAL seed guard'
        T = fit_temperature(vl, vt)
        aff = R.fit_evidence_affine(vl, vt, num_classes=K, prior_per_class=prior,
                                    scale_init=R.read_evidence_affine(model.head)[0],
                                    bias_init=R.read_evidence_affine(model.head)[1])

        ref = evaluate_episodic(model=model, test_iterable=it, ood_pools=pools,
                                num_classes=K, interpretation=interp,
                                ece_bins=int(cfg.eval.ece_bins),
                                temperature=(T if interp == 'softmax' else None),
                                prior_per_class=prior, device=DEVICE, logger=None)['summary']
        it2 = EpisodicIterableDataset(test_split, n_way=K, k_shot=int(cfg.dataset.k_shot),
                                      q_query=int(cfg.dataset.q_query),
                                      num_episodes=N_SMOKE, seed_offset=0)
        new = R.factorial_evaluate(model, cfg, test_seeds=list(range(N_SMOKE)),
                                   ood_pools=pools, device=DEVICE, temperature=T,
                                   affine_valfit=aff, prior_per_class=prior,
                                   ece_bins=int(cfg.eval.ece_bins), logits_out=None,
                                   log_every=0)

        native = R.native_score_name(interp)
        old_sc = 'vacuity' if interp == 'evidential' else 'msp'
        pset = 'evidential_native' if interp == 'evidential' else 'softmax'
        pairs = []
        for k, v in ref.items():
            if k.startswith(('ood_auroc__', 'fpr_at_95_tpr__')):
                h, pool, sc = k.split('__')
                if sc == old_sc:
                    pairs.append((k, v, new.get(f'{h}__{pool}__{native}')))
            elif k in ('accuracy_mean', 'ece_pooled', 'brier_mean', 'f1_macro_mean'):
                pairs.append((k, v, new.get(f'{k}__{pset}')))
            elif k == 'ece_ts':
                pairs.append((k, v, new.get('ece_pooled__ts')))
        # Three tiers, matching rq_core.regression_guard's reasoning: a real
        # logic error moves these by >=1e-2, four orders above the float32
        # noise floor, so 1e-6 separates them without ever calling hardware
        # noise a defect. Anything non-exact is still PRINTED -- a silent
        # tolerance is how a genuine regression hides.
        TOL = 1e-6
        n_exact = sum(1 for _k, a, b in pairs if b is not None and a == b)
        bad = [(k, a, b) for k, a, b in pairs if b is None or abs(a - b) > TOL]
        noisy = [(k, a, b) for k, a, b in pairs
                 if b is not None and a != b and abs(a - b) <= TOL]
        print(f'  [1] T1.6 {tag:<11}: {n_exact}/{len(pairs)} exact'
              + (f', {len(noisy)} within {TOL:g}' if noisy else '')
              + (f', {len(bad)} MISMATCHED' if bad else ''))
        for k, a, b in noisy:
            print(f'        ~ {k}: |diff|={abs(a - b):.2e}  (float32 noise, ok)')
        for k, a, b in bad:
            print(f'        ! {k}: {a!r} vs {b!r}')
        if bad:
            failures.append(f'T1.6 {tag}: {len(bad)} beyond {TOL:g}')
        return model, cfg, new, T, aff, prior, K

    model_e, cfg_e2, new_e, T_e, aff_e, prior_e, K_e = _smoke(cfg_e, 'evidential')
    model_s, cfg_s2, new_s, T_s, aff_s, prior_s, K_s = _smoke(cfg_s, 'softmax')

    # [2] T1.5 -- all scores present for BOTH interpretations.
    for tag, summ in (('evidential', new_e), ('softmax', new_s)):
        missing = [s for s in R.FACTORIAL_SCORES
                   if f'ood_auroc__svhn_far__{s}' not in summ]
        if missing:
            failures.append(f'T1.5 {tag}: missing {missing}')
    print(f'  [2] T1.5 all-scores-both-arms: '
          f'{"PASS" if not any(f.startswith("T1.5") for f in failures) else "FAIL"}')

    # [3] T1.7 -- energy is logsumexp of the RAW logits in both arms.
    probe = torch.randn(64, K_e, device=DEVICE) * 3.0
    got = R.all_id_scores(probe, model_e.head, num_classes=K_e, temperature=T_e,
                          prior_per_class=prior_e, affine_valfit=aff_e)['energy']
    want = torch.logsumexp(probe, dim=-1)
    d17 = float((got - want).abs().max())
    print(f'  [3] T1.7 energy==logsumexp(raw logits): max|diff|={d17:.3e}')
    if d17 != 0.0:
        failures.append('T1.7: energy is not logsumexp of raw logits')

    # [4] T1.8 / T2.6 -- determinism.
    aff_again = R.fit_evidence_affine(
        *R.load_val_logits(model_e, cfg_e2, DEVICE, REPO)[:2],
        num_classes=K_e, prior_per_class=prior_e,
        scale_init=R.read_evidence_affine(model_e.head)[0],
        bias_init=R.read_evidence_affine(model_e.head)[1])
    same_fit = (aff_again == aff_e)
    new_e2 = R.factorial_evaluate(
        model_e, cfg_e2, test_seeds=list(range(N_SMOKE)),
        ood_pools=R.build_ood_pools(model_e, cfg_e2, DEVICE), device=DEVICE,
        temperature=T_e, affine_valfit=aff_e, prior_per_class=prior_e,
        ece_bins=int(cfg_e2.eval.ece_bins), logits_out=None, log_every=0)
    same_eval = (json.dumps(new_e, sort_keys=True) == json.dumps(new_e2, sort_keys=True))
    print(f'  [4] T1.8/T2.6 determinism: affine_fit={same_fit}  eval_json={same_eval}')
    if not (same_fit and same_eval):
        failures.append('T1.8/T2.6: non-deterministic')

    # [5] T5.6 + the closed-form parameter budget.
    sweep_cells = S.build_sweep_configs(REPO, heads=RQ5_HEADS, ranks=RQ5_RANKS,
                                        seeds=RQ5_SEEDS)
    guard = S.assert_only_rank_and_seed_differ(REPO, sweep_cells)
    print(f'  [5] T5.6 sweep configs: {guard["n_configs"]} configs, '
          f'{guard["n_keys_checked"]} keys checked, ok={guard["ok"]}')
    if not guard['ok']:
        failures.append(f'T5.6: {guard["offending_keys"]}')
    from src.utils import count_trainable_params
    for rank in RQ5_RANKS:
        c = [x for x in sweep_cells if x['rank'] == rank][0]
        n = int(count_trainable_params(build_model(load_config(REPO / c['config']))))
        want_n = S.expected_trainable_params(rank)
        ok = (n == want_n)
        print(f'        rank {rank:>2}: {n:>7} params (formula {want_n:>7}) {"ok" if ok else "MISMATCH"}')
        if not ok:
            failures.append(f'param formula mismatch at rank {rank}')
    if S.expected_trainable_params(16) != 31746:
        failures.append('rank-16 budget != 31,746 (the committed grid value)')

    print()
    if failures:
        print('SELF-TEST FAILED:'); [print('  -', f) for f in failures]
        raise SystemExit('Fix the above before running Phase A.')
    print('SELF-TEST PASSED -- the diagonal is provably untouched; safe to proceed.')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 164MB/s]


  [1] T1.6 evidential : 12/12 exact
  [1] T1.6 softmax    : 13/13 exact
  [2] T1.5 all-scores-both-arms: PASS
  [3] T1.7 energy==logsumexp(raw logits): max|diff|=0.000e+00
  [4] T1.8/T2.6 determinism: affine_fit=True  eval_json=True
  [5] T5.6 sweep configs: 21 configs, 47 keys checked, ok=True
        rank  1:    2886 params (formula    2886) ok
        rank  2:    4810 params (formula    4810) ok
        rank  4:    8658 params (formula    8658) ok
        rank  8:   16354 params (formula   16354) ok
        rank 16:   31746 params (formula   31746) ok
        rank 32:   62530 params (formula   62530) ok
        rank 64:  124098 params (formula  124098) ok

SELF-TEST PASSED -- the diagonal is provably untouched; safe to proceed.


## 6. Phase A — RQ1 factorial + RQ2 affine refit (T1 + T2)

For every selected grid cell this: loads the trained adapter, fits **both**
post-hoc parameters on the **frozen VAL seeds only** (temperature `T`, and the
evidence affine `(scale, bias)`), then runs the 600 test episodes once and
scores them under **every** (objective, score) combination.

Two guards run per cell and are recorded in the output JSON:

- **VAL-seed tripwire (T2.5)** — fitting aborts unless the seeds it is about to
  use are exactly `10000..10099` *and* provably disjoint from the 600 test
  seeds. This protects the project's single most important scientific convention.
- **Regression guard (T1.6)** — the native-interpretation keys are diffed
  against the committed `results/grid/*_metrics.json`, and graded in three
  tiers. `exact` = bit-for-bit. `within_tol` (max |diff| <= 1e-6) is also a
  **pass**: it is float32 accumulation differing across hardware. A full
  600-episode re-run of `cifar_1shot/mbnet/lora` on CPU against the committed
  T4 numbers came back at **max |diff| = 2.2e-7**, i.e. Step 10 reproduces to
  the 7th decimal off its original hardware. Only `MISMATCH` (> 1e-6) is a
  real problem — a genuine logic error moves AUROC by 1e-2 or more.

A cell with no checkpoint is logged as `no_checkpoint` and skipped (with
`ALLOW_RETRAIN=False`), so the 21-cell `cifar_fs/5shot` gap costs nothing now
and can be filled by a later session with `ALLOW_RETRAIN=True` — `--resume`
semantics mean nothing already done is repeated.

Output: one JSON per cell in `results/rq_factorial/`, plus (if
`PERSIST_LOGITS`) the raw per-episode logits in `results/rq_logits/`. That dump
is the structural fix for the problem Phase 0 just measured: with logits on
disk, every future post-hoc scoring question is a re-analysis, not a retrain.

In [11]:
if not RUN_PHASE_A:
    print('RUN_PHASE_A = False -- skipped.')
else:
    grid = json.load(open(REPO / 'configs/grid/_index.json'))['cells']
    sel = [c for c in grid if all(c.get(k) == v for k, v in CELL_FILTER.items())]
    sel.sort(key=lambda c: (c['priority'], c['dataset'], c['k_shot'],
                            c['backbone'], c['adapter'], c['head'], c['seed']))
    have = sum(1 for c in sel if (REPO / c['checkpoint']).exists())
    log(f'[A] {len(sel)} cell(s) selected; {have} have a checkpoint, '
        f'{len(sel) - have} would need training (ALLOW_RETRAIN={ALLOW_RETRAIN})')
    if not ALLOW_RETRAIN and have == 0:
        log('[A] nothing to do: no checkpoints and retraining is off. Either '
            'attach the Step 10 artifacts (Section 4) or set ALLOW_RETRAIN=True.')
    else:
        est = have * 3 + (0 if not ALLOW_RETRAIN else (len(sel) - have) * 21)
        log(f'[A] rough estimate: ~{est} min ({est / 60:.1f} h) at ~3 min/eval, '
            f'~21 min/train+eval')
        counts = D.run_phase_a(
            REPO, sel, device=DEVICE,
            out_dir=REPO / 'results' / 'rq_factorial',
            logits_dir=(REPO / 'results' / 'rq_logits') if PERSIST_LOGITS else None,
            num_episodes=NUM_EPISODES, allow_retrain=ALLOW_RETRAIN,
            wandb_mode=WANDB_MODE, max_minutes=MAX_MINUTES_A, log=log)

        # Surface the regression guard across every cell done so far -- a
        # single MISMATCH invalidates the comparison to Step 10 and must not
        # be discovered later in a table.
        recs = A.load_records(REPO / 'results' / 'rq_factorial')
        st = {}
        for r in recs:
            s = r['regression_guard'].get('status', '?')
            st[s] = st.get(s, 0) + 1
        log(f'[A] regression-guard status across {len(recs)} cell(s): {st}')
        bad = [r for r in recs if r['regression_guard'].get('status') == 'MISMATCH']
        for r in bad:
            log(f"    MISMATCH {r['config']}: "
                f"max|diff|={r['regression_guard']['max_abs_diff']:.3e} "
                f"at {r['regression_guard']['max_abs_diff_key']}")

[A] 120 cell(s) selected; 99 have a checkpoint, 21 would need training (ALLOW_RETRAIN=False)
[A] rough estimate: ~297 min (5.0 h) at ~3 min/eval, ~21 min/train+eval
[A] (1/120) cifar_fs_5shot_mobilenetv3_small_bottleneck_parallel_evidential_seed42
    no checkpoint and ALLOW_RETRAIN=False -> skipped
[A] (2/120) cifar_fs_5shot_mobilenetv3_small_bottleneck_parallel_evidential_seed43
    no checkpoint and ALLOW_RETRAIN=False -> skipped
[A] (3/120) cifar_fs_5shot_mobilenetv3_small_bottleneck_parallel_evidential_seed44
    no checkpoint and ALLOW_RETRAIN=False -> skipped
[A] (4/120) cifar_fs_5shot_mobilenetv3_small_bottleneck_parallel_softmax_seed42
    no checkpoint and ALLOW_RETRAIN=False -> skipped
[A] (5/120) cifar_fs_5shot_mobilenetv3_small_bottleneck_parallel_softmax_seed43
    no checkpoint and ALLOW_RETRAIN=False -> skipped
[A] (6/120) cifar_fs_5shot_mobilenetv3_small_bottleneck_parallel_softmax_seed44
    no checkpoint and ALLOW_RETRAIN=False -> skipped
[A] (7/120) cifar_fs_5shot_m

100%|██████████| 9.83M/9.83M [00:00<00:00, 95.0MB/s]


    T=0.7460  affine native=(1.0000, 0.0000)  refit=(5.3372, -13.5490)
    ep 200/600  acc=0.8726
    ep 400/600  acc=0.8777
    ep 600/600  acc=0.8776
    guard: exact (13/13 exact, max|diff|=0.00e+00)  [794.0s]
[A] (11/120) cifar_fs_5shot_mobilenetv3_small_lora_softmax_seed43
    T=0.7232  affine native=(1.0000, 0.0000)  refit=(5.0036, -13.9590)
    ep 200/600  acc=0.8725
    ep 400/600  acc=0.8772
    ep 600/600  acc=0.8777
    guard: exact (13/13 exact, max|diff|=0.00e+00)  [186.2s]
[A] (12/120) cifar_fs_5shot_mobilenetv3_small_lora_softmax_seed44
    T=0.6917  affine native=(1.0000, 0.0000)  refit=(5.5139, -14.1136)
    ep 200/600  acc=0.8819
    ep 400/600  acc=0.8843
    ep 600/600  acc=0.8860
    guard: exact (13/13 exact, max|diff|=0.00e+00)  [187.9s]
[A] (13/120) cifar_fs_5shot_resnet18_bottleneck_parallel_evidential_seed42
    no checkpoint and ALLOW_RETRAIN=False -> skipped
[A] (14/120) cifar_fs_5shot_resnet18_bottleneck_parallel_evidential_seed43
    no checkpoint and ALLO

## 7. Phase B — RQ5 bottleneck-rank sweep (T5.3–T5.6)

The existing four budget points confound *budget* with *adapter type* and with
*which weights train*. This holds dataset (CIFAR-FS), backbone (ResNet-18),
shot count and adapter family (parallel 1x1 bottleneck) fixed and moves **only**
`adapter.rank` — 2,886 to 124,098 trainable parameters, a 43x span, with rank 16
passing exactly through the Step 10 point.

`REUSE_GRID_RANK16` matters: rank 16 + parallel + ResNet-18 + CIFAR-FS 5-shot
*is* the Step 10 recipe (verified — `n_params` 31,746 both ways). If those
checkpoints were recovered, retraining them would burn ~18 min each to
reproduce a model already on disk.

This is also the experiment that settles RQ3's causal ambiguity (T3.4): it
varies budget with backbone **and** architecture fixed, which the Step 10 grid
structurally cannot do.

In [12]:
if not RUN_PHASE_B:
    print('RUN_PHASE_B = False -- skipped.')
else:
    sweep_cells = S.build_sweep_configs(REPO, heads=RQ5_HEADS, ranks=RQ5_RANKS,
                                        seeds=RQ5_SEEDS)
    guard = S.assert_only_rank_and_seed_differ(REPO, sweep_cells)
    log(f'[B] {len(sweep_cells)} sweep configs; T5.6 guard ok={guard["ok"]}')
    if not guard['ok']:
        raise SystemExit(f'T5.6 FAILED -- the sweep would be confounded: '
                         f'{guard["offending_keys"]}')

    done = sum(1 for c in sweep_cells
               if (REPO / 'results' / 'rq5' / f"{c['cell']}.json").exists())
    log(f'[B] {done} already done; ~{(len(sweep_cells) - done) * 21} min '
        f'({(len(sweep_cells) - done) * 21 / 60:.1f} h) estimated for the rest')

    counts = D.run_phase_b(
        REPO, sweep_cells, device=DEVICE,
        out_dir=REPO / 'results' / 'rq5',
        logits_dir=(REPO / 'results' / 'rq5_logits') if PERSIST_LOGITS else None,
        num_episodes=NUM_EPISODES, wandb_mode=WANDB_MODE,
        max_minutes=MAX_MINUTES_B, reuse_grid_rank16=REUSE_GRID_RANK16, log=log)

RUN_PHASE_B = False -- skipped.


## 8. Aggregate — the tables the RQs are actually asking for

Reads only the per-cell JSONs, so this cell is safe to re-run locally later
without a GPU. Everything it prints is also written to
`results/rq_summary.json`.

**Read the `balanced` flag on the eta-squared output.** The formula is exact
for a balanced design and quietly stops being exact otherwise — a silently
wrong variance attribution is exactly the failure mode T4.1 flagged in the
existing decomposition, so it is reported rather than assumed.

In [13]:
import numpy as np

summary_out = {}
recs = A.load_records(REPO / 'results' / 'rq_factorial')
print(f'RQ1/RQ2 records: {len(recs)}')

if recs:
    rows = A.factorial_observations(recs)
    summary_out['rq1'] = {'n_records': len(recs), 'n_observations': len(rows)}

    for grp in ('far', 'near'):
        t = A.table_2x4(rows, grp)
        if all(np.isnan(t[o][s]['mean']) for o in t for s in A.RQ1_SCORES):
            continue
        print(f'\n=== RQ1 -- 2x4 objective x score, mean AUROC ({grp}-OOD) ===')
        print(f"  {'trained as':<14}" + ''.join(f'{s:>17}' for s in A.RQ1_SCORES))
        for obj in ('evidential', 'softmax'):
            print(f'  {obj:<14}' + ''.join(
                f"{t[obj][s]['mean']:>17.4f}" for s in A.RQ1_SCORES))
        n = t['evidential'][A.RQ1_SCORES[0]]['n']
        print(f'  (n={n} observations per cell)')
        summary_out.setdefault('rq1_tables', {})[grp] = t

    verdict = A.rq1_verdict(rows)
    summary_out['rq1_verdict'] = verdict
    print('\n=== RQ1 -- variance attribution (eta^2) ===')
    for grp, v in verdict.items():
        e = v['eta_squared']
        print(f'  {grp}-OOD  (n={v["n"]}, balanced={v["balanced"]})')
        for k in sorted(e, key=lambda k: -e[k]):
            print(f'      {k:<22} {e[k]:>7.3f}')
        print(f'      -> DOMINANT AXIS: {v["dominant"]} '
              f'(score/objective = {v["ratio_score_over_objective"]:.2f}x)')

    r2 = A.rq2_table(recs)
    if r2:
        print('\n=== RQ2 -- evidence-affine refit, before -> after ===')
        print(f"  {'cell':<44}{'ECE b':>8}{'ECE a':>8}{'d':>8}"
              f"{'AUROC d':>10}{'rho min':>9}")
        for row in sorted(r2, key=lambda r: r['cell']):
            ad = [v for k, v in row.items() if k.startswith('auroc_delta__')]
            rh = [v for k, v in row.items() if k.startswith('rank_rho__')]
            print(f"  {row['cell'][:43]:<44}{row['ece_before']:>8.4f}"
                  f"{row['ece_after']:>8.4f}{row['ece_delta']:>8.4f}"
                  f"{np.mean(ad):>10.4f}{min(rh):>9.4f}")
        print('\n  fitted (scale, bias) vs the frozen grid constants (2, -6):')
        for row in sorted(r2, key=lambda r: r['cell'])[:12]:
            print(f"    {row['cell'][:40]:<42} trained="
                  f"({row['affine_trained'][0]:.3f}, {row['affine_trained'][1]:.3f})"
                  f"  refit=({row['affine_refit'][0]:.3f}, {row['affine_refit'][1]:.3f})")
        v2 = A.rq2_verdict(r2)
        summary_out['rq2_verdict'] = v2
        summary_out['rq2_rows'] = r2
        print('\n  RQ2 verdict:', json.dumps(v2, indent=2))
        print('  Reading it: ECE improving is only interesting if AUROC survives.'
              '\n  `reordering_ever_observed` answers T2.8 empirically -- a'
              '\n  per-logit affine is monotone, but vacuity depends on the SUM'
              '\n  over classes, so reordering is possible in principle.')

# ---------------------------------------------------------------- RQ5
rq5 = A.load_records(REPO / 'results' / 'rq5')
print(f'\nRQ5 records: {len(rq5)}')
if rq5:
    curve = A.rq5_curve(rq5)
    summary_out['rq5'] = curve
    print('\n=== RQ5 -- metric vs trainable-parameter budget (architecture FIXED) ===')
    print(f"  {'rank':>5}{'params':>10}{'seeds':>7}{'acc':>9}{'ECE evid':>10}"
          f"{'ECE soft':>10}{'AUROC far':>11}{'AUROC near':>12}")
    for r in curve['curve']:
        def f(k):
            return f"{r[k]:.4f}" if r.get(k) is not None else '    -   '
        print(f"  {r['rank']:>5}{r['n_params']:>10}{r['n_seeds']:>7}"
              f"{f('acc_mean'):>9}{f('ece_evid_mean'):>10}{f('ece_softmax_mean'):>10}"
              f"{f('auroc_far_mean'):>11}{f('auroc_near_mean'):>12}")
    print(f"\n  best accuracy at rank {curve['best_accuracy_rank']}, "
          f"best ECE at rank {curve['best_ece_rank']}")
    print(f"  budgets differ: {curve['budgets_differ']}   "
          f"ECE optimum interior: {curve['ece_optimum_is_interior']}")

    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    c = curve['curve']
    xs = [r['n_params'] for r in c]
    fig, ax1 = plt.subplots(figsize=(7.5, 4.5))
    acc = [r['acc_mean'] for r in c]
    ece = [r['ece_evid_mean'] for r in c]
    ax1.plot(xs, acc, 'o-', color='tab:blue', label='accuracy')
    ax1.set_xscale('log'); ax1.set_xlabel('trainable parameters (log)')
    ax1.set_ylabel('accuracy', color='tab:blue')
    ax1.tick_params(axis='y', labelcolor='tab:blue')
    ax2 = ax1.twinx()
    ax2.plot(xs, ece, 's--', color='tab:red', label='ECE (evidential)')
    ax2.set_ylabel('ECE', color='tab:red')
    ax2.tick_params(axis='y', labelcolor='tab:red')
    for r in c:
        ax1.annotate(f"r{r['rank']}", (r['n_params'], r['acc_mean']),
                     textcoords='offset points', xytext=(0, 7), fontsize=8)
    plt.title('RQ5: accuracy and calibration vs budget\n'
              '(CIFAR-FS 5-shot, ResNet-18, parallel bottleneck — only rank varies)')
    fig.tight_layout()
    out_png = REPO / 'results' / 'rq5_rank_sweep.png'
    fig.savefig(out_png, dpi=140)
    print(f'  wrote {out_png}')
    from IPython.display import Image, display
    display(Image(filename=str(out_png)))

with open(REPO / 'results' / 'rq_summary.json', 'w') as f:
    json.dump(summary_out, f, indent=2, sort_keys=True, default=str)
print('\nwrote results/rq_summary.json')

RQ1/RQ2 records: 99

=== RQ1 -- 2x4 objective x score, mean AUROC (far-OOD) ===
  trained as                  msp           energy           ts_msp   vacuity_valfit
  evidential               0.7855           0.9107           0.7732           0.9052
  softmax                  0.7896           0.9291           0.7735           0.9255
  (n=96 observations per cell)

=== RQ1 -- 2x4 objective x score, mean AUROC (near-OOD) ===
  trained as                  msp           energy           ts_msp   vacuity_valfit
  evidential               0.7839           0.8306           0.7732           0.8240
  softmax                  0.7696           0.8266           0.7564           0.8126
  (n=96 observations per cell)

=== RQ1 -- variance attribution (eta^2) ===
  far-OOD  (n=792, balanced=True)
      score                    0.437
      residual                 0.242
      design                   0.232
      design:score             0.060
      design:objective         0.025
      objective        

## 9. Pack + push artifacts

Two zips on purpose. The **results** zip is small and always pushable; the
**logits** zip can be ~1.5 GB for a full grid, and a failed push of that must
not take the results with it.

The session log travels with the results — the Step 11 incident was only
diagnosable because a log survived, and it did so by luck.

In [14]:
import glob, hashlib, zipfile

def _pack(zip_path, files, stem):
    files = [p for p in files if os.path.exists(p)]
    if not files:
        print(f'  {stem}: nothing to pack'); return None
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in files:
            zf.write(p)
        zf.writestr('MANIFEST.txt', '\n'.join(
            f'{p}  {os.path.getsize(p)}B  '
            f'sha256={hashlib.sha256(open(p, "rb").read()).hexdigest()}'
            for p in files))
    mb = os.path.getsize(zip_path) / 1e6
    print(f'  wrote {zip_path} ({len(files)} files, {mb:.1f} MB)')
    return zip_path

WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else str(REPO.parent)

results_files = (
    sorted(glob.glob('results/rq_factorial/*')) +
    sorted(glob.glob('results/rq5/*')) +
    ['results/rq_checkpoint_audit.json', 'results/rq_summary.json',
     'results/new_rqs_session.log', 'results/rq5_rank_sweep.png'] +
    sorted(glob.glob('configs/rq5/*.yaml')) + ['configs/rq5/_index.json'] +
    ['rq_core.py', 'rq5_sweep.py', 'rq_drivers.py', 'rq_aggregate.py']
)
zip_results = _pack(f'{WORK}/new_rqs_results.zip', results_files, 'results')
zip_logits = _pack(f'{WORK}/new_rqs_logits.zip',
                   sorted(glob.glob('results/rq_logits/*.npz')) +
                   sorted(glob.glob('results/rq5_logits/*.npz')), 'logits')

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = _s.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = _s.get_secret('KAGGLE_KEY')
    HAVE_SECRETS = True
except Exception:
    HAVE_SECRETS = False

for zp, stem in ((zip_results, 'new-rqs-results'), (zip_logits, 'new-rqs-logits')):
    if zp is None:
        continue
    if HAVE_SECRETS:
        ds_dir = f'{WORK}/{stem}_dataset'
        os.makedirs(ds_dir, exist_ok=True)
        subprocess.run(['cp', zp, ds_dir], check=True)
        json.dump({'title': stem,
                   'id': f"{os.environ['KAGGLE_USERNAME']}/{stem}",
                   'licenses': [{'name': 'CC0-1.0'}]},
                  open(f'{ds_dir}/dataset-metadata.json', 'w'))
        r = subprocess.run(['kaggle', 'datasets', 'create', '-p', ds_dir, '-q'],
                           capture_output=True, text=True)
        if r.returncode != 0:
            subprocess.run(['kaggle', 'datasets', 'version', '-p', ds_dir,
                            '-m', 'update', '-q'])
        print(f'  pushed Kaggle dataset: {stem}')
    else:
        from IPython.display import FileLink, display
        display(FileLink(zp))
print('\nNo Kaggle Secrets -> use the download links above.' if not HAVE_SECRETS
      else '\nPushed. These survive the tab closing.')

  wrote /kaggle/working/new_rqs_results.zip (129 files, 0.3 MB)
  wrote /kaggle/working/new_rqs_logits.zip (99 files, 1043.5 MB)


/kaggle/working/new_rqs_results.zip

/kaggle/working/new_rqs_logits.zip


No Kaggle Secrets -> use the download links above.


## After this session

1. **Check the regression guard first.** In `results/rq_summary.json` (and the
   Section 6 output) every cell's `regression_guard.status` should be `exact`.
   Anything else means the comparison to Step 10 is not clean — investigate
   before quoting a single number from this run.
2. **Merge the zip locally** into `results/rq_factorial/`, `results/rq5/`, then
   re-run Section 8's aggregation locally to regenerate the committed tables.
3. **Promote the code.** `rq_core.py` / `rq5_sweep.py` / `rq_drivers.py` /
   `rq_aggregate.py` are in the results zip. T1.1 asks for `_id_score_set` in
   `src/evaluators/episodic.py` to be refactored onto an explicit score list —
   `rq_core.all_id_scores` is that function, already validated against the old
   branch. Moving it in-tree (with `tests/test_factorial_scores.py` built from
   Section 5's checks) closes T1.1/T1.5–T1.8 properly.
4. **Record the T0 verdict** in `docs/NEW_RQS_TASK_PLAN.md` (T0.3) and
   `progress.txt`, using `results/rq_checkpoint_audit.json` as the evidence.
5. **T0.4 is now partly handled**: `results/rq_logits/*.npz` means future
   post-hoc scoring never needs a retrain. Also drop `--keep-checkpoints` from
   any run whose outputs feed a future RQ.
6. **Transcribe from the JSON, never from this notebook's stdout** — the repo
   convention since Step 9.
7. Tick the T0/T1/T2/T5 boxes in `docs/NEW_RQS_TASK_PLAN.md` as they land.

### What this notebook deliberately does NOT do
- T3 / T4 (no compute needed — run locally off `results/mvt_results.json`).
- T5.1's citation check, T3.3's ECCV paper, T4.3's arXiv:2308.11838 — literature
  work, not compute.
- T6 (propagating the new RQs through the repo) — do that after T1/T2 land.